# 06 · Pre-Haiyan nighttime-light baseline characterization

This notebook describes the **observability, brightness, and temporal variation**
of reliability-qualified DNB-BRDF nighttime lights before Haiyan. It moves from
regional context to individual Tacloban pixels. Post-Haiyan values in the final
pixel figure are shown only to contrast them with the fixed pre-Haiyan median.
The main STL impact scenarios and departures are organized in Notebook 07.

## 1. Data and fixed reference support

Keep the original source stores, reference period, quality rules, and fixed pixel supports.

In [ ]:
from pathlib import Path
import json
import hashlib
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import geopandas as gpd
from rasterio.enums import Resampling
from rasterio.features import rasterize
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm.auto import tqdm

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_DIR / "datasets"
PROCESSED_DIR = DATA_DIR / "VNP46/processed"
BASELINE_STORE = PROCESSED_DIR / "baseline/Haiyan_VNP46A2_baseline.zarr"
EXISTING_STORE = PROCESSED_DIR / "Haiyan_VNP46A2.zarr"
GHSL_PATH = next(DATA_DIR.glob("**/GHSL_SMOD_E2015.tif"))
MUNICIPALITIES_PATH = DATA_DIR / "boundaries/MuniCities/MuniCities.shp"
OUTPUT_DIR = PROJECT_DIR / "output/baseline_characterization_v2"
FIGURE_DIR, TABLE_DIR = OUTPUT_DIR / "figures", OUTPUT_DIR / "tables"
for path in (FIGURE_DIR,TABLE_DIR): path.mkdir(parents=True,exist_ok=True)
EVENT_DATE = pd.Timestamp("2013-11-08")
BASELINE_DAYS, AGGREGATION_DAYS, KERNEL_SIZE = 60, 4, 5
BASELINE_START = EVENT_DATE-pd.Timedelta(days=BASELINE_DAYS)
PRE_EVENT_END = EVENT_DATE-pd.Timedelta(days=1)
CHARACTERIZATION_START = pd.Timestamp("2012-01-19")
CHARACTERIZATION_END = PRE_EVENT_END
GHSL_MASKS, SETTLEMENT_MASK = {"G3":(22,23,30)}, "G3"
SPATIAL_COMPLETENESS_PCT = 10.0
RQ_CLIP_PERCENTILE = 95.0
MIN_BASELINE_OBSERVATIONS = {1:5,4:3}
KNOWN_FILL_VALUES = (-9999.,-999.,-32768.,6553.5,65535.)
REBUILD_PROFILES = False  # True if source pixel values change without a date-axis change.
DNB_BAND, MQF_BAND = "DNB_BRDF_Corrected_NTL", "Mandatory_Quality_Flag"
GAP_FILLED_BAND = "Gap_Filled_DNB_BRDF_Corrected_NTL"

PLOT_FONT, PLOT_TEXT_COLOR, PLOT_GRID_COLOR = "Arial", "#243B5A", "#E8EDF3"
EVENT_COLOR, BASELINE_COLOR = "#0057FF", "#6C7882"
DNB_COLOR, GAP_FILLED_COLOR = "#0091FF", "#FF0000"
RQ_COLOR, RQ_FOUR_DAY_COLOR = "#00C54F", "#009227"
NTL_DISPLAY_MIN, NTL_DISPLAY_MAX = 0., 5.  # Display only; never truncate analytical values.
GHSL_CLASS_COLORS = {11:"#cdf57a",12:"#abcd66",13:"#375623",21:"#ffff00",22:"#a87000",23:"#732600",30:"#ff0000"}
GHSL_CLASS_LABELS = {11:"Very low density rural",12:"Low density rural",13:"Rural cluster",21:"Suburban / peri-urban",22:"Semi-dense urban",23:"Dense urban",30:"Urban centre"}
LOCATION_COLORS = {"Tacloban":"#0072B2","Palo":"#E69F00","Guiuan":"#009E73","Alangalang":"#CC79A7","Ormoc":"#D55E00","Baybay":"#56B4E9"}
SC_GREEN_COLORSCALE = [[0.,"rgba(255,255,255,0)"],[.2,"rgba(220,242,215,.25)"],
    [.4,"rgba(166,219,160,.45)"],[.6,"rgba(90,174,110,.65)"],[.8,"rgba(25,130,70,.82)"],[1.,"rgba(0,88,43,.98)"]]

def style_figure(fig,title,width=1280,height=720):
    fig.update_layout(template="plotly_white",paper_bgcolor="rgba(0,0,0,0)",plot_bgcolor="white",
        width=width,height=height,font=dict(family=PLOT_FONT,size=14,color=PLOT_TEXT_COLOR),
        title=dict(text=title,x=.03,xanchor="left",font=dict(size=22)),
        margin=dict(l=85,r=85,t=95,b=130),
        legend=dict(orientation="h",x=0,y=-.18,xanchor="left",yanchor="top"))
    fig.update_xaxes(zeroline=False,gridcolor=PLOT_GRID_COLOR)
    fig.update_yaxes(zeroline=False,gridcolor=PLOT_GRID_COLOR)
    return fig

def finish_figure(fig,filename):
    fig.write_html(FIGURE_DIR/f"{filename}.html",include_plotlyjs="directory")
    fig.show()

def add_haiyan_marker(fig,row=None,col=None):
    fig.add_vline(x=EVENT_DATE.to_pydatetime(),line_dash="dash",line_color=EVENT_COLOR,line_width=2,row=row,col=col)


In [ ]:
early = xr.open_zarr(BASELINE_STORE,chunks={},consolidated=False,mask_and_scale=False)
existing = xr.open_zarr(EXISTING_STORE,chunks={},consolidated=False,mask_and_scale=False)
if not np.all(early.processed.values == 1): raise RuntimeError("Finish Notebook 05 first.")
for axis in ("x","y"):
    np.testing.assert_allclose(early[axis].values,existing[axis].values,rtol=0,atol=1e-7)
early=early.assign_coords(x=existing.x,y=existing.y)
early["spatial_ref"]=existing["spatial_ref"]
complete=existing.processed.values==1
if len(pd.DatetimeIndex(existing.date.values[~complete]).difference(early.date.values)):
    raise RuntimeError("Existing event-period processing is unfinished.")
existing=existing.isel(date=np.flatnonzero(complete))
new_dates=pd.DatetimeIndex(early.date.values).difference(existing.date.values)
a2=xr.concat([early.sel(date=new_dates),existing],dim="date",data_vars="minimal",coords="minimal",compat="equals",join="exact").sortby("date")
a2=a2.rio.write_crs(existing.attrs["crs_wkt"])
template=a2[DNB_BAND].isel(date=0,drop=True)
ghsl=rxr.open_rasterio(GHSL_PATH,masked=True).squeeze("band",drop=True)
ghsl=ghsl.rio.reproject_match(template,resampling=Resampling.nearest).values
municipalities=gpd.read_file(MUNICIPALITIES_PATH)
region=municipalities[municipalities.PROVINCE.str.strip().str.title().isin(
    ["Samar","Eastern Samar","Northern Samar","Leyte","Southern Leyte"])].to_crs(a2.rio.crs).copy()
region["unit_name"]=region.NAME_2.str.replace("City of ","",regex=False).str.replace(" City","",regex=False).str.strip()
region["zone_id"]=np.arange(1,len(region)+1)
zone=rasterize([(g,int(i)) for g,i in zip(region.geometry,region.zone_id)],out_shape=ghsl.shape,
               transform=template.rio.transform(recalc=True),fill=0)
land=(zone>0)&np.isin(ghsl,list(GHSL_CLASS_LABELS))
land_pixels=np.flatnonzero(land)
ghsl_vector=ghsl.ravel()[land_pixels]
g3_vector=np.isin(ghsl_vector,GHSL_MASKS[SETTLEMENT_MASK])

poi_table=pd.DataFrame([
    ("Tacloban",125.0015,11.2434),("Palo",124.9904,11.1577),("Guiuan",125.7232,11.0312),
    ("Alangalang",124.8465,11.2071),("Ormoc",124.6068,11.0088),("Baybay",124.7989,10.6766)
],columns=["unit_name","longitude","latitude"])
landmarks=pd.DataFrame([
    ("Airport vicinity",125.0278,11.2275),("Port vicinity",125.0060,11.2500),
    ("Main-road vicinity",125.0030,11.2420),("Residential vicinity",125.0010,11.2650),
    ("Plaza vicinity",125.0050,11.2450),("Astrodome vicinity",125.0010,11.2240)
],columns=["unit_name","longitude","latitude"])
profiles={"Regional G3":np.flatnonzero(g3_vector)}
meta=[dict(profile="Regional G3",kind="region",class_code="G3",extent="region",color=RQ_FOUR_DAY_COLOR)]
for value,label in GHSL_CLASS_LABELS.items():
    name=f"{value} · {label}"; profiles[name]=np.flatnonzero(ghsl_vector==value)
    meta.append(dict(profile=name,kind="GHSL",class_code=str(value),extent="region",color=GHSL_CLASS_COLORS[value]))
for table,kind in [(poi_table,"centre"),(landmarks,"landmark")]:
    for point in table.itertuples():
        ix=int(abs(a2.x.values-point.longitude).argmin()); iy=int(abs(a2.y.values-point.latitude).argmin())
        for size in ([KERNEL_SIZE] if kind=="centre" else [1,3,5]):
            local=np.zeros(land.shape,dtype=bool); r=size//2
            local[max(0,iy-r):iy+r+1,max(0,ix-r):ix+r+1]=True
            if kind=="centre":
                parent_ids=region.loc[region.unit_name.eq(point.unit_name),"zone_id"].values
                local &= np.isin(zone,parent_ids)
            indices=np.flatnonzero(local.ravel()[land_pixels]&g3_vector)
            name=f"{point.unit_name} | {size}×{size}"
            profiles[name]=indices
            meta.append(dict(profile=name,kind=kind,class_code="G3",extent=f"{size}×{size}",
                color=LOCATION_COLORS.get(point.unit_name,"#174A7E")))
profile_meta=pd.DataFrame(meta)
print(f"A2: {a2.sizes['date']} available dates; {len(land_pixels):,} regional land pixels.")


In [ ]:
dates=pd.DatetimeIndex(a2.date.values)
day_blocks=np.asarray((dates-EVENT_DATE).days//AGGREGATION_DAYS)
blocks=np.arange(day_blocks.min(),day_blocks.max()+1)
cache_settings={"version":3,"stores":[str(BASELINE_STORE.resolve()),str(EXISTING_STORE.resolve())],
    "dates":dates.strftime("%Y-%m-%d").tolist(),"p95":RQ_CLIP_PERCENTILE,"reference":str(BASELINE_START),
    "minimum_reference_composites":MIN_BASELINE_OBSERVATIONS[4],"fill_values":KNOWN_FILL_VALUES,
    "aggregation_days":AGGREGATION_DAYS,"ghsl_mtime":GHSL_PATH.stat().st_mtime,
    "boundaries_mtime":MUNICIPALITIES_PATH.stat().st_mtime,
    "masks":{p:idx.tolist() for p,idx in profiles.items()}}
cache_key=hashlib.sha256(json.dumps(cache_settings,sort_keys=True).encode()).hexdigest()
cache_dir=OUTPUT_DIR/"cache"/cache_key; cache_dir.mkdir(parents=True,exist_ok=True)

def median(values,axis=None):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore",RuntimeWarning)
        return np.nanmedian(values,axis=axis)

def read_days(indices):
    batch=a2[[DNB_BAND,MQF_BAND,GAP_FILLED_BAND]].isel(date=indices).compute()
    raw=batch[DNB_BAND].values.reshape(len(indices),-1)[:,land_pixels].astype("float32")
    quality=batch[MQF_BAND].values.reshape(len(indices),-1)[:,land_pixels]
    gap=batch[GAP_FILLED_BAND].values.reshape(len(indices),-1)[:,land_pixels].astype("float32")
    raw[(quality!=0)|~np.isfinite(raw)|(raw<0)|np.isin(raw,KNOWN_FILL_VALUES)]=np.nan
    gap[~np.isfinite(gap)|(gap<0)|np.isin(gap,KNOWN_FILL_VALUES)]=np.nan
    capped=raw.copy()
    for j in range(len(indices)):
        good=raw[j,g3_vector]; good=good[np.isfinite(good)]
        if good.size: capped[j]=np.minimum(raw[j],np.percentile(good,RQ_CLIP_PERCENTILE))
    return raw,capped,gap

if not REBUILD_PROFILES and (cache_dir/"complete.json").exists():
    daily=pd.read_csv(cache_dir/"daily.csv",parse_dates=["date"])
    four_day=pd.read_csv(cache_dir/"four_day.csv",parse_dates=["date"])
    ntl0=np.load(cache_dir/"reference.npy"); reference_count=np.load(cache_dir/"reference_count.npy")
else:
    reference_cache={}; reference_composites=[]
    for block in range(-BASELINE_DAYS//AGGREGATION_DAYS,0):
        indices=np.flatnonzero(day_blocks==block)
        if len(indices):
            arrays=read_days(indices); reference_cache[block]=arrays
            reference_composites.append(median(arrays[1],axis=0))
        else: reference_composites.append(np.full(len(land_pixels),np.nan,dtype="float32"))
    reference_composites=np.asarray(reference_composites)
    reference_count=np.isfinite(reference_composites).sum(axis=0)
    ntl0=median(reference_composites,axis=0)
    reference_ok=(reference_count>=MIN_BASELINE_OBSERVATIONS[4])&np.isfinite(ntl0)&(ntl0>0)
    fixed={p:idx[reference_ok[idx]] for p,idx in profiles.items()}
    def summarize(arrays,date):
        rows=[]
        for name,idx in fixed.items():
            fresh,capped,gap=[a[idx] for a in arrays]
            valid=np.isfinite(capped); n=int(valid.sum()); sc=100*n/len(idx) if len(idx) else np.nan
            ratio=100*capped[valid].sum()/ntl0[idx][valid].sum() if n else np.nan
            rows.append((date,name,len(idx),n,sc,median(fresh),median(capped),median(gap),ratio))
        return rows
    daily_rows=[]; block_rows=[]
    for block in tqdm(blocks,desc="Observed daily and 4-day profiles"):
        indices=np.flatnonzero(day_blocks==block)
        if len(indices):
            arrays=reference_cache[block] if block in reference_cache else read_days(indices)
            for j,index in enumerate(indices): daily_rows.extend(summarize([a[j] for a in arrays],dates[index]))
            composite=[median(a,axis=0) for a in arrays]
        else: composite=[np.full(len(land_pixels),np.nan)]*3
        midpoint=EVENT_DATE+pd.Timedelta(days=int(block)*4+1.5)
        block_rows.extend(summarize(composite,midpoint))
    columns=["date","profile","fixed_pixels","valid_pixels","spatial_coverage_pct","raw_uncapped","rq_median","gap_median","reference_pct"]
    daily=pd.DataFrame(daily_rows,columns=columns); four_day=pd.DataFrame(block_rows,columns=columns)
    daily.to_csv(cache_dir/"daily.csv",index=False); four_day.to_csv(cache_dir/"four_day.csv",index=False)
    np.save(cache_dir/"reference.npy",ntl0); np.save(cache_dir/"reference_count.npy",reference_count)
    (cache_dir/"complete.json").write_text(json.dumps({"key":cache_key}))
reference_ok=(reference_count>=MIN_BASELINE_OBSERVATIONS[4])&np.isfinite(ntl0)&(ntl0>0)
fixed={p:idx[reference_ok[idx]] for p,idx in profiles.items()}
profile_meta["base_pixels"]=profile_meta.profile.map(lambda p:len(profiles[p]))
profile_meta["fixed_pixels"]=profile_meta.profile.map(lambda p:len(fixed[p]))
for frame,days in [(daily,1),(four_day,4)]:
    frame["aggregation_days"]=days
    frame["observed"]=frame.spatial_coverage_pct.ge(SPATIAL_COMPLETENESS_PCT)&frame.valid_pixels.gt(0)
    frame["raw_median_ntl"]=frame.rq_median.where(frame.observed)
    frame["normalized_ntl_pct"]=frame.reference_pct.where(frame.observed)
    frame["reference_start"],frame["reference_end"]=BASELINE_START,PRE_EVENT_END
    frame["sc_threshold_pct"],frame["p95"]=SPATIAL_COMPLETENESS_PCT,RQ_CLIP_PERCENTILE
    frame["date_start"]=frame.date-pd.to_timedelta((days-1)/2,unit="D")
    frame["date_end"]=frame.date_start+pd.to_timedelta(days-1,unit="D")
    frame["observation_status"]=np.where(frame.observed,"observed","not observable")
    frame["class_code"]=frame.profile.map(profile_meta.set_index("profile").class_code)
    frame["extent"]=frame.profile.map(profile_meta.set_index("profile").extent)
print("Fixed support — zero-pixel profiles remain visibly unavailable:")
display(profile_meta[["profile","base_pixels","fixed_pixels"]])


## 2. Spatial baseline context

The maps locate the settlement classes and pre-Haiyan reference radiance before comparing their time series.

The GHSL and NTL panels use the same geography. Blank terrestrial cells indicate unavailable reference values, not zero light.

In [ ]:
# ------------------------------------------------------------
# Display grids — analytical values remain unchanged
# ------------------------------------------------------------

map_classes = [
    (10, "Water", "#B8D9EF"),
    *[
        (value, GHSL_CLASS_LABELS[value], GHSL_CLASS_COLORS[value])
        for value in GHSL_CLASS_LABELS
    ],
]

ghsl_grid = np.full(land.shape, np.nan)

for index, (value, _, _) in enumerate(map_classes):
    mask = (ghsl == value)
    if value != 10:
        mask &= land
    ghsl_grid[mask] = index

reference_grid = np.full(land.size, np.nan)
reference_grid[land_pixels[reference_ok]] = ntl0[reference_ok]
reference_grid = reference_grid.reshape(land.shape)

# Water is shown only where GHSL identifies water.
# Missing terrestrial NTL remains blank.
water_grid = np.where(ghsl == 10, 1.0, np.nan)

ghsl_scale = [
    [stop, color]
    for index, (_, _, color) in enumerate(map_classes)
    for stop in (
        index / len(map_classes),
        (index + 1) / len(map_classes),
    )
]

west, south, east, north = region.total_bounds
regional_bounds = (
    west - 0.03,
    south - 0.03,
    east + 0.03,
    north + 0.03,
)
tacloban_bounds = (124.94, 11.14, 125.10, 11.32)

# Dissolve internal municipality boundaries.
outline = region.dissolve().geometry.iloc[0].boundary
outline_x, outline_y = [], []

for line in (
    list(outline.geoms)
    if hasattr(outline, "geoms")
    else [outline]
):
    xx, yy = line.xy
    outline_x.extend([*xx, None])
    outline_y.extend([*yy, None])


# ------------------------------------------------------------
# Two large regional maps; Tacloban insets at their top right
# ------------------------------------------------------------

fig = go.Figure()
style_figure(fig, "", 1700, 1050)

panels = [
    # axis, grid, bounds, x-domain, y-domain, inset
    (1, ghsl_grid, regional_bounds, [0.015, 0.345], [0.22, 0.95], False),
    (2, reference_grid, regional_bounds, [0.515, 0.845], [0.22, 0.95], False),
    (3, ghsl_grid, tacloban_bounds, [0.305, 0.490], [0.62, 0.925], True),
    (4, reference_grid, tacloban_bounds, [0.805, 0.990], [0.62, 0.925], True),
]

for number, grid, bounds, x_domain, y_domain, inset in panels:
    is_ghsl = number in (1, 3)
    xref = "x" if number == 1 else f"x{number}"
    yref = "y" if number == 1 else f"y{number}"
    xaxis_name = "xaxis" if number == 1 else f"xaxis{number}"
    yaxis_name = "yaxis" if number == 1 else f"yaxis{number}"

    xi = np.flatnonzero(
        (a2.x.values >= bounds[0])
        & (a2.x.values <= bounds[2])
    )
    yi = np.flatnonzero(
        (a2.y.values >= bounds[1])
        & (a2.y.values <= bounds[3])
    )

    xx = a2.x.values[xi]
    yy = a2.y.values[yi]

    if not is_ghsl:
        fig.add_trace(go.Heatmap(
            x=xx,
            y=yy,
            z=water_grid[np.ix_(yi, xi)],
            xaxis=xref,
            yaxis=yref,
            colorscale=[[0, "#B8D9EF"], [1, "#B8D9EF"]],
            zmin=0,
            zmax=1,
            showscale=False,
            hoverinfo="skip",
        ))

    fig.add_trace(go.Heatmap(
        x=xx,
        y=yy,
        z=grid[np.ix_(yi, xi)],
        xaxis=xref,
        yaxis=yref,
        colorscale=ghsl_scale if is_ghsl else "Inferno",
        zmin=-0.5 if is_ghsl else NTL_DISPLAY_MIN,
        zmax=len(map_classes) - 0.5 if is_ghsl else NTL_DISPLAY_MAX,
        showscale=(number == 2),
        zsmooth=False,
        hoverongaps=False,
        hoverinfo="skip" if is_ghsl else None,
        hovertemplate=(
            None if is_ghsl else
            "%{y:.4f}°N, %{x:.4f}°E"
            "<br>NTL: %{z:.3f} nW cm⁻² sr⁻¹<extra></extra>"
        ),
        colorbar=dict(
            orientation="h",
            title=dict(
                text="DNB-BRDF nighttime lights (nW cm⁻² sr⁻¹)",
                side="top",
            ),
            x=0.735,
            xanchor="center",
            y=0.11,
            yanchor="middle",
            len=0.36,
            thickness=18,
            outlinewidth=0,
            tickvals=np.linspace(NTL_DISPLAY_MIN, NTL_DISPLAY_MAX, 6),
        ),
    ))

    # Outer outline only: fine enough to preserve raster visibility.
    fig.add_trace(go.Scatter(
        x=outline_x,
        y=outline_y,
        xaxis=xref,
        yaxis=yref,
        mode="lines",
        line=dict(color="rgba(36,59,90,0.28)", width=0.4),
        showlegend=False,
        hoverinfo="skip",
    ))

    fig.update_layout({
        xaxis_name: dict(
            domain=x_domain,
            anchor=yref,
            range=[bounds[0], bounds[2]],
            constrain="domain",
            showgrid=False,
            zeroline=False,
            showticklabels=not inset,
            ticks="" if inset else "outside",
            tickvals=np.arange(
                np.ceil(bounds[0]), np.floor(bounds[2]) + 1, 1
            ),
            tickformat=".0f",
            ticksuffix="°E",
            title=None,
        ),
        yaxis_name: dict(
            domain=y_domain,
            anchor=xref,
            range=[bounds[1], bounds[3]],
            scaleanchor=xref,
            scaleratio=1 / np.cos(np.deg2rad(
                (bounds[1] + bounds[3]) / 2
            )),
            constrain="domain",
            showgrid=False,
            zeroline=False,
            showticklabels=not inset,
            ticks="" if inset else "outside",
            tickvals=np.arange(
                np.ceil(bounds[1]), np.floor(bounds[3]) + 1, 1
            ),
            tickformat=".0f",
            ticksuffix="°N",
            title=None,
        ),
    })

    if inset:
        # Frame and labels only inside the inset.
        fig.add_shape(
            type="rect",
            xref=xref,
            yref=yref,
            x0=bounds[0],
            x1=bounds[2],
            y0=bounds[1],
            y1=bounds[3],
            line=dict(color=PLOT_TEXT_COLOR, width=1),
            fillcolor="rgba(0,0,0,0)",
        )

        visible = poi_table[
            poi_table.longitude.between(bounds[0], bounds[2])
            & poi_table.latitude.between(bounds[1], bounds[3])
        ]

        fig.add_trace(go.Scatter(
            x=visible.longitude,
            y=visible.latitude,
            text=visible.unit_name,
            xaxis=xref,
            yaxis=yref,
            mode="markers+text",
            marker=dict(
                color="white",
                size=5,
                line=dict(color=PLOT_TEXT_COLOR, width=0.8),
            ),
            textposition="top center",
            textfont=dict(size=11, color=PLOT_TEXT_COLOR),
            showlegend=False,
        ))

    else:
        # Locate the inset footprint without six overlapping regional labels.
        fig.add_shape(
            type="rect",
            xref=xref,
            yref=yref,
            x0=tacloban_bounds[0],
            x1=tacloban_bounds[2],
            y0=tacloban_bounds[1],
            y1=tacloban_bounds[3],
            line=dict(color="#64748B", width=1, dash="dot"),
            fillcolor="rgba(0,0,0,0)",
        )


# ------------------------------------------------------------
# Panel headings — no main title
# ------------------------------------------------------------

for x, text in [
    (0.18, "GHSL settlement context"),
    (0.68, "60-day pixel baseline"),
]:
    fig.add_annotation(
        x=x, y=0.995,
        xref="paper", yref="paper",
        text=f"{text}",
        showarrow=False,
        font=dict(size=19, color=PLOT_TEXT_COLOR),
    )

for x in (0.3975, 0.8975):
    fig.add_annotation(
        x=x, y=0.945,
        xref="paper", yref="paper",
        text="Tacloban",
        showarrow=False,
        font=dict(size=14, color=PLOT_TEXT_COLOR),
    )


# ------------------------------------------------------------
# Fixed GHSL legend: two columns × four rows, including water
# ------------------------------------------------------------

for index, (value, label, color) in enumerate(map_classes):
    column, row = divmod(index, 4)
    x = 0.055 + column * 0.235
    y = 0.165 - row * 0.042

    fig.add_shape(
        type="rect",
        xref="paper", yref="paper",
        x0=x, x1=x + 0.008,
        y0=y - 0.006, y1=y + 0.006,
        fillcolor=color,
        line=dict(width=0),
    )
    fig.add_annotation(
        x=x + 0.013, y=y,
        xref="paper", yref="paper",
        text=f"{value} · {label}",
        showarrow=False,
        xanchor="left",
        yanchor="middle",
        font=dict(size=17, color=PLOT_TEXT_COLOR),
    )

fig.update_layout(
    title_text="",
    showlegend=False,
    margin=dict(l=55, r=35, t=35, b=25),
)

finish_figure(fig, "01_context_and_reference")

The regional class map identifies where dense settlement supports the baseline comparisons. Class membership is spatial context, not an impact estimate.

In [ ]:
class_grid = np.full(land.shape, np.nan)
for j, code in enumerate([22, 23, 30]):
    class_grid[land & (ghsl == code)] = j

class_colors = [
    GHSL_CLASS_COLORS[code]
    for code in [22, 23, 30]
]
class_scale = [
    [stop, color]
    for j, color in enumerate(class_colors)
    for stop in (j/3, (j+1)/3)
]

west, south, east, north = region.total_bounds
pad = .04
bounds = (west-pad, south-pad, east+pad, north+pad)

xi = np.flatnonzero(
    (a2.x.values >= bounds[0]) & (a2.x.values <= bounds[2])
)
yi = np.flatnonzero(
    (a2.y.values >= bounds[1]) & (a2.y.values <= bounds[3])
)

fig = go.Figure()
fig.add_trace(go.Heatmap(
    x=a2.x.values[xi],
    y=a2.y.values[yi],
    z=class_grid[np.ix_(yi, xi)],
    colorscale=class_scale,
    zmin=-.5, zmax=2.5,
    showscale=False,
    hoverongaps=False,
    hovertemplate="Class index: %{z}<extra></extra>",
))

# Regional exterior only: internal lines would obscure the pixels.
outline = region.dissolve().geometry.iloc[0].boundary
outline_x, outline_y = [], []
for line in (
    list(outline.geoms)
    if hasattr(outline, "geoms") else [outline]
):
    x, y = line.xy
    outline_x.extend([*x, None])
    outline_y.extend([*y, None])

fig.add_trace(go.Scatter(
    x=outline_x, y=outline_y,
    mode="lines",
    line=dict(color="#52677D", width=.8),
    opacity=.45,
    showlegend=False,
    hoverinfo="skip",
))

for code in [22, 23, 30]:
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode="markers",
        marker=dict(
            symbol="square", size=14,
            color=GHSL_CLASS_COLORS[code],
        ),
        name=f"{code} · {GHSL_CLASS_LABELS[code]}",
    ))

style_figure(fig, "", 1100, 900)
fig.update_layout(
    title=None,
    font=dict(size=17),
    margin=dict(l=65, r=40, t=30, b=110),
    legend=dict(
        orientation="h", x=0, y=-.13,
        font=dict(size=17),
    ),
)
fig.update_xaxes(
    range=[bounds[0], bounds[2]],
    nticks=4, ticksuffix="°E",
    showgrid=False, title_text=None,
)
fig.update_yaxes(
    range=[bounds[1], bounds[3]],
    nticks=5, ticksuffix="°N",
    showgrid=False, title_text=None,
    scaleanchor="x",
    scaleratio=1/np.cos(np.deg2rad((south+north)/2)),
)
finish_figure(fig, "baseline_regional_ghsl_context")

## 3. Pixel brightness and temporal variability

Compare pixel-level median radiance with median absolute deviation (MAD) across three processing stages. Each panel uses the same supported pixels; source, daily RQ, and four-day RQ values remain distinct.

In [ ]:
# Same pre-Haiyan pixels, three processing stages:
# 1. Source DNB-BRDF: no MQF, P95, or SC gate
# 2. Daily RQ: MQF == 0, daily G3 P95 cap, regional daily SC gate
# 3. Four-day RQ: median of capped daily pixels, regional 4D SC gate

baseline_calendar = pd.date_range(
    CHARACTERIZATION_START, PRE_EVENT_END, freq="D"
)
pre_idx = np.flatnonzero(
    (dates >= CHARACTERIZATION_START) & (dates <= PRE_EVENT_END)
)
pre_dates = dates[pre_idx]

# Consider all regional land pixels. The common-support rule below
# determines which pixels appear in every comparison panel.
pixel_columns = np.arange(len(land_pixels))
flat_pixels = land_pixels[pixel_columns]
pixel_classes = ghsl_vector[pixel_columns]
n_pixels = len(pixel_columns)

def pixel_median_and_mad(values):
    n_observed = np.isfinite(values).sum(axis=0)
    centre = np.full(values.shape[1], np.nan)
    spread = np.full(values.shape[1], np.nan)

    for start in range(0, values.shape[1], 5000):
        sl = slice(start, start + 5000)
        part = values[:, sl]
        centre[sl] = median(part, axis=0)
        spread[sl] = median(np.abs(part - centre[sl]), axis=0)

    return centre, spread, n_observed

daily_values = np.full(
    (len(pre_idx), n_pixels), np.nan, dtype=np.float32
)

# Stage 1: source DNB-BRDF, without quality or SC filtering.
for start in tqdm(range(0, len(pre_idx), 24), desc="Source DNB-BRDF"):
    batch_idx = pre_idx[start:start + 24]
    values = (
        a2[DNB_BAND].isel(date=batch_idx).compute().values
        .reshape(len(batch_idx), -1)[:, flat_pixels]
        .astype(np.float32)
    )
    values[
        ~np.isfinite(values)
        | (values < 0)
        | np.isin(values, KNOWN_FILL_VALUES)
    ] = np.nan
    daily_values[start:start + len(batch_idx)] = values

raw_median, raw_mad, raw_n = pixel_median_and_mad(daily_values)

# Replace source values with MQF-qualified, P95-capped daily pixels.
# Keep all dates for now: 4D composites are formed before the 4D SC gate.
for start in tqdm(range(0, len(pre_idx), 24), desc="Qualified DNB-BRDF"):
    batch_idx = pre_idx[start:start + 24]
    daily_values[start:start + len(batch_idx)] = (
        read_days(batch_idx)[1][:, pixel_columns]
    )

regional4 = pd.read_csv(
    cache_dir / "four_day.csv",
    usecols=[
        "date", "profile", "valid_pixels",
        "spatial_coverage_pct",
    ],
    parse_dates=["date"],
)
regional4 = regional4.loc[
    regional4["profile"].eq("Regional G3")
].copy()
regional4["date_start"] = regional4["date"] - pd.Timedelta(days=1.5)
regional4["date_end"] = regional4["date"] + pd.Timedelta(days=1.5)
regional4["observed"] = (
    regional4["spatial_coverage_pct"].ge(
        SPATIAL_COMPLETENESS_PCT
    )
    & regional4["valid_pixels"].gt(0)
)
regional4 = regional4.loc[
    regional4["date_start"].ge(CHARACTERIZATION_START)
    & regional4["date_end"].le(PRE_EVENT_END)
].sort_values("date")

# Stage 3: four-day pixel composites on qualified regional blocks.
block_values = np.full(
    (len(regional4), n_pixels), np.nan, dtype=np.float32
)
pre_block_ids = day_blocks[pre_idx]

for j, block in enumerate(regional4.itertuples()):
    if not block.observed:
        continue

    block_id = int(
        (block.date_start - EVENT_DATE).days // AGGREGATION_DAYS
    )
    rows = np.flatnonzero(pre_block_ids == block_id)

    if len(rows):
        block_values[j] = median(daily_values[rows], axis=0)

four_median, four_mad, four_n = pixel_median_and_mad(block_values)

# Stage 2: apply the daily regional SC gate to daily RQ pixels.
regional_daily = pd.read_csv(
    cache_dir / "daily.csv",
    usecols=[
        "date", "profile", "valid_pixels",
        "spatial_coverage_pct",
    ],
    parse_dates=["date"],
)
regional_daily = regional_daily.loc[
    regional_daily["profile"].eq("Regional G3")
    & regional_daily["date"].between(
        CHARACTERIZATION_START, PRE_EVENT_END
    )
]
qualified_dates = pd.DatetimeIndex(
    regional_daily.loc[
        regional_daily["spatial_coverage_pct"].ge(
            SPATIAL_COMPLETENESS_PCT
        )
        & regional_daily["valid_pixels"].gt(0),
        "date",
    ]
)
daily_values[~pre_dates.isin(qualified_dates)] = np.nan

rq_median, rq_mad, rq_n = pixel_median_and_mad(daily_values)
del daily_values, block_values

# Require the same pixel to have sufficient support in all three stages.
minimum_daily = max(
    3, int(np.ceil(.10 * len(baseline_calendar)))
)
minimum_four = max(
    3, int(np.ceil(.10 * len(regional4)))
)

common = (
    (raw_n >= minimum_daily)
    & (rq_n >= minimum_daily)
    & (four_n >= minimum_four)
    & np.isfinite(raw_median) & np.isfinite(raw_mad)
    & np.isfinite(rq_median) & np.isfinite(rq_mad)
    & np.isfinite(four_median) & np.isfinite(four_mad)
)

Read each point as one pixel. The spread of MAD at a given median shows temporal variability; differences among panels also reflect quality filtering and four-day compositing.

In [ ]:
stages = [
    ("Source DNB-BRDF",
     raw_median, raw_mad, raw_n, len(baseline_calendar)),
    ("Daily RQ NTL",
     rq_median, rq_mad, rq_n, len(baseline_calendar)),
    ("RQ NTL (4D)",
     four_median, four_mad, four_n, len(regional4)),
]

fig = make_subplots(
    rows=1, cols=3,
    horizontal_spacing=.055,
    subplot_titles=[stage[0] for stage in stages],
)

for col, (_, levels, mads, counts, denominator) in enumerate(stages, 1):
    for code, label in GHSL_CLASS_LABELS.items():
        selected = common & (pixel_classes == code)

        fig.add_trace(go.Scattergl(
            x=levels[selected],
            y=mads[selected],
            mode="markers",
            name=f"{code} · {label}",
            legendgroup=str(code),
            showlegend=(col == 1),
            marker=dict(
                size=6,
                color=GHSL_CLASS_COLORS[code],
                opacity=.30,
            ),
            customdata=np.column_stack([
                counts[selected],
                100 * counts[selected] / denominator,
            ]),
            hovertemplate=(
                f"GHSL {code} · {label}"
                "<br>Pixel median: %{x:.2f}"
                "<br>Temporal MAD: %{y:.2f}"
                "<br>Observed: %{customdata[0]:.0f}"
                " (%{customdata[1]:.1f}%)"
                "<extra></extra>"
            ),
        ), row=1, col=col)

    fig.update_xaxes(
        title_text="Pixel median (nW cm⁻² sr⁻¹)",
        rangemode="tozero",
        tickfont=dict(size=15),
        title_font=dict(size=16),
        row=1, col=col,
    )
    fig.update_yaxes(
        title_text="Temporal MAD (nW cm⁻² sr⁻¹)" if col == 1 else "",
        rangemode="tozero",
        tickfont=dict(size=15),
        title_font=dict(size=16),
        row=1, col=col,
    )

style_figure(fig, "", 1200, 600)
fig.update_layout(
    title=None,
    margin=dict(l=100, r=45, t=75, b=165),
    legend=dict(
        orientation="h",
        x=0, y=-.20,
        font=dict(size=15),
    ),
)
fig.update_annotations(font=dict(size=18, color=PLOT_TEXT_COLOR))
finish_figure(fig, "pixel_median_mad_three_processing_stages")

print(
    f"{common.sum():,} identical pixels shown in each panel; "
    f"daily baseline: {CHARACTERIZATION_START:%d %b %Y}–"
    f"{PRE_EVENT_END:%d %b %Y}."
)

## 4. Regional G3 baseline and observability

The regional series establishes the pre-Haiyan level and makes missing daily and four-day observations visible. The zero-value checks are diagnostics of observability, not an instruction to discard low values.

In [ ]:

display_start = CHARACTERIZATION_START.to_period("M").to_timestamp()
month_starts = pd.date_range(
    display_start, CHARACTERIZATION_END, freq="MS"
)
quarter_ticks = pd.date_range(
    display_start, CHARACTERIZATION_END, freq="3MS"
)


The green strip reports spatial completeness. Interpret the NTL line only where qualified observations exist; gaps are part of the baseline evidence.

In [ ]:
calendar = pd.date_range(dates.min(), dates.max(), freq="D")
regional = (
    daily.loc[daily.profile.eq("Regional G3")]
    .set_index("date").reindex(calendar)
)
regional4 = (
    four_day.loc[four_day.profile.eq("Regional G3")]
    .sort_values("date")
)

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[.09, .91], vertical_spacing=.06,
)
style_figure(fig, "", 1600, 500)

fig.add_trace(go.Heatmap(
    x=calendar, y=["Daily SC"], z=[regional.spatial_coverage_pct],
    colorscale=SC_GREEN_COLORSCALE, zmin=0, zmax=100,
    colorbar=dict(
        title=dict(text="SC (%)", font=dict(size=12)),
        tickfont=dict(size=12), len=.2, x = 0.95, y=.98, thickness=15,
    ),
    hovertemplate="%{x|%d %b %Y}<br>Daily SC: %{z:.1f}%<extra></extra>",
    hoverongaps=False,
), row=1, col=1)

# Draw uncapped observations first, behind the summaries.
fig.add_trace(go.Scatter(
    x=calendar, y=regional.raw_uncapped,
    mode="markers", name="Daily DNB-BRDF NTL",
    marker=dict(size=5, color='gray', opacity=.50),
    hovertemplate="%{x|%d %b %Y}<br>Daily DNB: %{y:.4f}<extra></extra>",
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=calendar, y=regional.gap_median,
    mode="lines", name="Gap-filled",
    line=dict(color=GAP_FILLED_COLOR, width=1.2),
    opacity=.55, connectgaps=False,
), row=2, col=1)


fig.add_trace(go.Scatter(
    x=regional4.date, y=regional4.raw_median_ntl,
    mode="lines", name="Reliability-Qualified NTL (4D) Median",
    line=dict(color='black', width=2.5, shape="hv"),
    marker=dict(size=5, line=dict(color="white", width=.6)),
    connectgaps=False,
    customdata=regional4[["spatial_coverage_pct", "valid_pixels"]],
    hovertemplate=(
        "%{x|%d %b %Y}<br>Four-day NTL: %{y:.4f}"
        "<br>Four-day SC: %{customdata[0]:.1f}%"
        "<br>Valid pixels: %{customdata[1]:.0f}<extra></extra>"
    ),
), row=2, col=1)

for row in (1, 2):
    add_haiyan_marker(fig, row, 1)

fig.update_layout(
    title=None, font=dict(size=17),
    margin=dict(l=40, r=40, t=40, b=40),
    legend=dict(
        orientation="h", x=0.45, y=0.94,
        xanchor="left", yanchor="top", font=dict(size=20),
    ),
)
fig.update_xaxes(
    tickfont=dict(size=16), tickformat="%b<br>%Y",
    dtick="M6", title_text=None,
)
fig.update_xaxes(showticklabels=False, row=1, col=1)
fig.update_yaxes(
    showticklabels=False, showgrid=False, title_text=None,
    row=1, col=1,
)

fig.update_yaxes(
    range=[0, 1.2], dtick=.3, tickfont=dict(size=17),
    title=dict(text="NTL (nW cm⁻² sr⁻¹)", font=dict(size=19)),
    row=2, col=1
)


# fig.update_yaxes(
#     title=dict(text="NTL (nW cm⁻² sr⁻¹)", font=dict(size=20)),
#     row=2, col=1, type="log"
# )

month_starts = pd.date_range(
    display_start, CHARACTERIZATION_END, freq="MS"
)

for month in month_starts:
    fig.add_vline(
        x=month,
        line_color=(
            "rgba(36,59,90,.20)"
            if month.month == 1
            else "rgba(36,59,90,.10)"
        ),
        line_width=1.2 if month.month == 1 else .8,
        layer="below",
        row=row, col=1,
    )

finish_figure(fig, "02_regional_observations")

In [ ]:
zero_daily = daily.loc[
    daily.profile.eq("Regional G3") & daily.raw_uncapped.eq(0),
    ["date", "valid_pixels", "fixed_pixels",
     "spatial_coverage_pct", "observed", "gap_median"]
].sort_values("spatial_coverage_pct", ascending=False)

zero_4d = four_day.loc[
    four_day.profile.eq("Regional G3") & four_day.raw_median_ntl.eq(0),
    ["date", "valid_pixels", "fixed_pixels", "spatial_coverage_pct"]
]

print(
    f"Daily zero medians: {len(zero_daily)}; "
    f"passing the {SPATIAL_COMPLETENESS_PCT:g}% SC gate: "
    f"{zero_daily.observed.sum()}"
)
print(f"Four-day RQ zero medians: {len(zero_4d)}")
display(zero_daily.head(20))

In [ ]:
regional_check = daily.loc[
    daily.profile.eq("Regional G3")
].sort_values("date").copy()

# Diagnostic threshold only; it does not alter the RQ series.
regional_check["near_zero"] = regional_check.raw_uncapped.le(0.05)
regional_check["SC_band"] = pd.cut(
    regional_check.spatial_coverage_pct,
    bins=[0, 10, 25, 50, 75, 101],
    labels=["<10%", "10–25%", "25–50%", "50–75%", "≥75%"],
    right=False,
)

by_coverage = regional_check.groupby(
    "SC_band", observed=False
).agg(
    days=("date", "size"),
    near_zero_days=("near_zero", "sum"),
)
by_coverage["near_zero_share_pct"] = (
    100 * by_coverage.near_zero_days / by_coverage.days
)
display(by_coverage.round(1))

near_zero_days = regional_check.loc[
    regional_check.near_zero,
    ["date", "raw_uncapped", "gap_median",
     "spatial_coverage_pct", "valid_pixels", "observed"]
].copy()
near_zero_days["days_since_previous"] = (
    near_zero_days.date.diff().dt.days
)
display(near_zero_days)

In [ ]:
regional_week = daily.loc[
    daily.profile.eq("Regional G3")
    & daily.date.between("2012-02-04", "2012-02-10")
].sort_values("date")

raw_week, _, _ = read_days(dates.get_indexer(regional_week.date))
support = fixed["Regional G3"]

rows = []
for j, day in enumerate(regional_week.itertuples()):
    pixels = raw_week[j, support]
    valid = np.isfinite(pixels)
    values = pixels[valid]

    rows.append({
        "date": day.date,
        "SC_pct": day.spatial_coverage_pct,
        "valid_pixels": valid.sum(),
        "daily_median": np.median(values) if len(values) else np.nan,
        "pixels_at_0.01_pct": (
            100 * np.isclose(values, 0.01, atol=1e-6).mean()
            if len(values) else np.nan
        ),
        "pixels_below_0.05_pct": (
            100 * (values <= 0.05).mean()
            if len(values) else np.nan
        ),
        "reference_median_of_observed_pixels": (
            np.median(ntl0[support][valid]) if valid.any() else np.nan
        ),
    })

display(pd.DataFrame(rows).round(2))

regional_4d = four_day.loc[
    four_day.profile.eq("Regional G3") & four_day.observed
]
print(
    "Four-day RQ medians ≤0.05:",
    regional_4d.raw_median_ntl.le(0.05).sum(),
    "of", len(regional_4d), "observed blocks"
)

In [ ]:
class_check = daily.loc[
    daily.date.between("2012-02-05", "2012-02-09")
    & daily.class_code.isin(["11", "12", "13", "21", "22", "23", "30"])
]

display(
    class_check.pivot(
        index="date", columns="class_code", values="raw_uncapped"
    ).round(3)
)
display(
    class_check.pivot(
        index="date", columns="class_code", values="spatial_coverage_pct"
    ).round(1)
)

The 7–90-day curves summarize observed daily RQ NTL with explicit minimum support. They are rolling medians, not STL trend windows; gaps indicate insufficient qualified days.

In [ ]:
# Regional Samar–Leyte G3: reliability-qualified NTL and exploratory STL
from statsmodels.tsa.seasonal import STL

regional_daily = pd.read_csv(
    cache_dir / "daily.csv",
    usecols=[
        "date", "profile", "raw_uncapped",
        "valid_pixels", "spatial_coverage_pct",
    ],
    parse_dates=["date"],
)
regional_daily = regional_daily.loc[
    regional_daily["profile"].eq("Regional G3")
    & regional_daily["date"].between(
        CHARACTERIZATION_START, CHARACTERIZATION_END
    )
].sort_values("date")

regional_blocks = pd.read_csv(
    cache_dir / "four_day.csv",
    usecols=[
        "date", "profile", "rq_median",
        "valid_pixels", "spatial_coverage_pct",
    ],
    parse_dates=["date"],
)
regional_blocks = regional_blocks.loc[
    regional_blocks["profile"].eq("Regional G3")
].copy()

# Include only four-day blocks wholly inside the pre-Haiyan period.
regional_blocks["date_start"] = (
    regional_blocks["date"] - pd.Timedelta(days=1.5)
)
regional_blocks["date_end"] = (
    regional_blocks["date"] + pd.Timedelta(days=1.5)
)
regional_blocks = regional_blocks.loc[
    regional_blocks["date_start"].ge(CHARACTERIZATION_START)
    & regional_blocks["date_end"].le(CHARACTERIZATION_END)
].sort_values("date")

calendar = pd.DatetimeIndex(regional_blocks["date"])
sc = pd.Series(
    regional_blocks["spatial_coverage_pct"].to_numpy(),
    index=calendar,
)
qualified = (
    regional_blocks["spatial_coverage_pct"].ge(
        SPATIAL_COMPLETENESS_PCT
    )
    & regional_blocks["valid_pixels"].gt(0)
)
rq = pd.Series(
    regional_blocks["rq_median"].where(qualified).to_numpy(),
    index=calendar,
)

if rq.notna().sum() < 14:
    raise ValueError("Too few observed regional G3 blocks for 28-day STL.")

# Missing blocks receive values solely to make a regular STL input.
# The observed RQ series remains unchanged.
stl_input = (
    rq.interpolate(method="time", limit_area="inside")
    .ffill()
    .bfill()
)
observed = rq.notna()
missing = ~observed
gap_groups = missing.ne(missing.shift(fill_value=False)).cumsum()
longest_gap = int(missing.groupby(gap_groups).sum().max())

display(pd.DataFrame([{
    "profile": "Regional G3",
    "calendar_blocks": len(calendar),
    "observed_RQ_blocks": int(observed.sum()),
    "blocks_supplied_to_STL": int(missing.sum()),
    "supplied_pct": round(100 * missing.mean(), 1),
    "longest_gap_blocks": longest_gap,
    "SC_threshold_pct": SPATIAL_COMPLETENESS_PCT,
}]))

# Seven four-day blocks represent a 28-day diagnostic component.
stl = STL(np.log1p(stl_input.to_numpy()), period=7, robust=True).fit()
trend = pd.Series(np.expm1(stl.trend), index=calendar)
seasonal = pd.Series(stl.seasonal, index=calendar)
residual = pd.Series(stl.resid, index=calendar)

fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    row_heights=[.25, .375, .225, .15],
    vertical_spacing=.055,
    subplot_titles=[
        "Observed nighttime lights",
        "Trend",
        "Seasonality · 28-day diagnostic",
        "Residual",
    ],
)

fig.add_trace(go.Scatter(
    x=regional_daily["date"],
    y=regional_daily["raw_uncapped"],
    mode="markers",
    marker=dict(color='graY', size=4, opacity=.38),
    name="Daily DNB-BRDF NTL",
    customdata=regional_daily[
        ["valid_pixels", "spatial_coverage_pct"]
    ].to_numpy(),
    hovertemplate=(
        "%{x|%d %b %Y}<br>Daily DNB-BRDF NTL: %{y:.2f}"
        "<br>Valid pixels: %{customdata[0]:.0f}"
        "<br>SC: %{customdata[1]:.0f}%<extra></extra>"
    ),
), row=1, col=1)

# The solid observed series covers this dashed input wherever RQ data exist.
# Visible dashed spans therefore identify blocks supplied to STL.
fig.add_trace(go.Scatter(
    x=calendar,
    y=stl_input,
    mode="lines",
    line=dict(color='gray', width=1.6, dash="dot", shape = "hv"),
    name="STL input",
    hovertemplate="%{x|%d %b %Y}<br>STL input: %{y:.2f}<extra></extra>",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=calendar,
    y=rq,
    mode="lines",
    connectgaps=False,
    line=dict(color='black', width=2.6, shape="hv"),
    name="RQ NTL (4D)",
    customdata=sc.to_numpy().reshape(-1, 1),
    hovertemplate=(
        "%{x|%d %b %Y}<br>RQ NTL (4D): %{y:.2f}"
        "<br>SC: %{customdata[0]:.0f}%<extra></extra>"
    ),
), row=1, col=1)

for row, values, color, label in [
    (2, trend, "#174A7E", "Trend"),
    (3, seasonal, "#E69F00", "Seasonality"),
    (4, residual, "#6C7882", "Residual"),
]:
    # Dashed portions show the decomposition through supplied input blocks.
    fig.add_trace(go.Scatter(
        x=calendar,
        y=values,
        mode="lines",
        line=dict(color=color, width=1.7, dash="dot"),
        opacity=.7,
        showlegend=False,
        hovertemplate=(
            f"%{{x|%d %b %Y}}<br>{label}: %{{y:.3f}}"
            "<br>STL model input<extra></extra>"
        ),
    ), row=row, col=1)

    fig.add_trace(go.Scatter(
        x=calendar,
        y=values.where(observed),
        mode="lines",
        connectgaps=False,
        line=dict(color=color, width=2.7),
        showlegend=False,
        hovertemplate=(
            f"%{{x|%d %b %Y}}<br>{label}: %{{y:.3f}}"
            "<br>Observed RQ block<extra></extra>"
        ),
    ), row=row, col=1)

for row in (3, 4):
    fig.add_hline(
        y=0, line_color="#AAB5C2", line_width=1,
        row=row, col=1,
    )

fig.update_yaxes(
    title_text="NTL (nW cm⁻² sr⁻¹)", rangemode="tozero",
    row=1, col=1,
)
fig.update_yaxes(
    title_text="Trend NTL<br>(nW cm⁻² sr⁻¹)", rangemode="tozero",
    row=2, col=1,
)
fig.update_yaxes(
    title_text="Seasonality<br>(log1p NTL)",
    row=3, col=1,
)
fig.update_yaxes(
    title_text="Residual<br>(log1p NTL)",
    row=4, col=1,
)

style_figure(fig, "", 1200, 1000)

for row in range(1, 5):
    fig.update_xaxes(
        range=[display_start, CHARACTERIZATION_END],
        showgrid=False,
        tickmode="array",
        tickvals=quarter_ticks,
        ticktext=[d.strftime("%b %Y") for d in quarter_ticks],
        tickfont=dict(size=14),
        showticklabels=(row == 4),
        row=row, col=1,
    )
    fig.update_yaxes(tickfont=dict(size=14), row=row, col=1)

    for month in month_starts:
        fig.add_vline(
            x=month,
            line_color=(
                "rgba(36,59,90,.20)"
                if month.month == 1
                else "rgba(36,59,90,.10)"
            ),
            line_width=1.2 if month.month == 1 else .8,
            layer="below",
            row=row, col=1,
        )

fig.update_layout(
    legend=dict(
        orientation="h", x=0, y=-.08,
        xanchor="left", yanchor="top",
        font=dict(size=16),
    ),
)
fig.update_annotations(font=dict(size=17, color=PLOT_TEXT_COLOR))
finish_figure(fig, "regional_g3_observed_and_stl")

In [ ]:
# Regional G3 · STL trend sensitivity to the assumed cycle length
# Daily input; the 28-day fit corresponds to the cycle used previously.

from statsmodels.tsa.seasonal import STL

regional_daily = pd.read_csv(
    cache_dir / "daily.csv",
    usecols=[
        "date", "profile", "rq_median",
        "valid_pixels", "spatial_coverage_pct",
    ],
    parse_dates=["date"],
)

regional_daily = regional_daily.loc[
    regional_daily["profile"].eq("Regional G3")
    & regional_daily["date"].between(
        CHARACTERIZATION_START, CHARACTERIZATION_END
    )
].sort_values("date")

calendar_daily = pd.date_range(
    CHARACTERIZATION_START, CHARACTERIZATION_END, freq="D"
)
regional_daily = regional_daily.set_index("date").reindex(calendar_daily)

sc_daily = regional_daily["spatial_coverage_pct"]
rq_daily = regional_daily["rq_median"].where(
    sc_daily.ge(SPATIAL_COMPLETENESS_PCT)
    & regional_daily["valid_pixels"].gt(0)
)

if rq_daily.notna().sum() < 30:
    raise ValueError("Too few observed daily RQ values for STL.")

# Supply missing days to obtain a regular daily STL input.
# This does not change the observed RQ series plotted below.
stl_input = (
    rq_daily.interpolate(method="time", limit_area="inside")
    .ffill()
    .bfill()
)

regional_four = four_day.loc[
    four_day["profile"].eq("Regional G3")
    & four_day["date_start"].ge(CHARACTERIZATION_START)
    & four_day["date_end"].le(CHARACTERIZATION_END)
].sort_values("date")

periods = {
    7:   ("Weekly · 7 d",       "#0072B2"),
    14:  ("Fortnightly · 14 d", "#009E73"),
    28:  ("Four-week · 28 d",   "#E69F00"),
    60:  ("Two-month · 60 d",   "#00A4AF"),
    90:  ("Three-month · 90 d", "#CC79A7"),
    120: ("Four-month · 120 d","#7B5EA7"),
}

trend_curves = {}

for period_days in periods:
    fit = STL(
        np.log1p(stl_input.to_numpy()),
        period=period_days,
        robust=True,
    ).fit()

    trend_curves[period_days] = pd.Series(
        np.expm1(fit.trend),
        index=calendar_daily,
    )

reference = trend_curves[28]

print(
    f"Observed RQ days: {rq_daily.notna().sum():,} / "
    f"{len(rq_daily):,}; days supplied to STL: "
    f"{rq_daily.isna().sum():,}"
)

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    row_heights=[.09, .64, .27],
    vertical_spacing=.06,
    subplot_titles=[
        "",
        "Four-day RQ NTL and daily STL trends",
        "Trend difference from the 28-day STL fit",
    ],
)

fig.add_trace(go.Heatmap(
    x=calendar_daily,
    y=["SC"],
    z=[sc_daily.to_numpy()],
    colorscale=SC_GREEN_COLORSCALE,
    zmin=0,
    zmax=100,
    hoverongaps=False,
    colorbar=dict(
        title=dict(text="SC (%)", font=dict(size=15)),
        tickfont=dict(size=14),
        tickvals=[0, 50, 100],
        thickness=16,
        len=.12,
        y=.98,
        x=1.01,
    ),
    hovertemplate=(
        "%{x|%d %b %Y}<br>Daily SC: %{z:.1f}%<extra></extra>"
    ),
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=regional_four["date"],
    y=regional_four["raw_median_ntl"],
    mode="lines",
    connectgaps=False,
    line=dict(
        color='black',
        width=2,
        shape="hv",
    ),
    name="RQ NTL (4D)",
    customdata=regional_four[
        ["spatial_coverage_pct", "valid_pixels"]
    ].to_numpy(),
    hovertemplate=(
        "%{x|%d %b %Y}<br>RQ NTL (4D): %{y:.2f}"
        "<br>SC: %{customdata[0]:.0f}%"
        "<br>Valid pixels: %{customdata[1]:.0f}"
        "<extra></extra>"
    ),
), row=2, col=1)

for period_days, (label, color) in periods.items():
    curve = trend_curves[period_days]

    # Longer assumed cycles are drawn more prominently.
    width = 1.5 + .017 * period_days

    fig.add_trace(go.Scatter(
        x=calendar_daily,
        y=curve,
        mode="lines",
        line=dict(color=color, width=width),
        name=f"STL · {label}",
        hovertemplate=(
            f"%{{x|%d %b %Y}}<br>{label}"
            "<br>STL trend: %{y:.2f}<extra></extra>"
        ),
    ), row=2, col=1)

    if period_days != 28:
        fig.add_trace(go.Scatter(
            x=calendar_daily,
            y=curve - reference,
            mode="lines",
            line=dict(
                color=color,
                width=1.2 + .012 * period_days,
            ),
            showlegend=False,
            hovertemplate=(
                f"%{{x|%d %b %Y}}<br>{label} − 28-day fit: "
                "%{y:+.2f}<extra></extra>"
            ),
        ), row=3, col=1)

fig.add_hline(
    y=0,
    row=3, col=1,
    line_color=PLOT_TEXT_COLOR,
    line_width=1,
)

first_month = calendar_daily.min().to_period("M").to_timestamp()
month_starts = pd.date_range(
    first_month, calendar_daily.max(), freq="MS"
)
quarter_ticks = pd.date_range(
    first_month, calendar_daily.max(), freq="3MS"
)

style_figure(fig, "", 1200, 900)

for row in (1, 2, 3):
    fig.update_xaxes(
        range=[CHARACTERIZATION_START, CHARACTERIZATION_END],
        tickmode="array",
        tickvals=quarter_ticks,
        ticktext=[d.strftime("%b %Y") for d in quarter_ticks],
        tickfont=dict(size=14),
        showticklabels=(row == 3),
        showgrid=False,
        row=row, col=1,
    )

for row in (2, 3):
    for month in month_starts:
        fig.add_vline(
            x=month,
            row=row, col=1,
            line_color=PLOT_GRID_COLOR,
            line_width=.8,
            opacity=.55,
            layer="below",
        )

    for month in month_starts:
        fig.add_vline(
            x=month,
            line_color=(
                "rgba(36,59,90,.20)"
                if month.month == 1
                else "rgba(36,59,90,.10)"
            ),
            line_width=1.2 if month.month == 1 else .8,
            layer="below",
            row=row, col=1,
            )
    fig.update_yaxes(tickfont=dict(size=14), row=row, col=1)

fig.update_yaxes(
    showticklabels=False,
    showgrid=False,
    row=1, col=1,
)
fig.update_yaxes(
    title_text="NTL (nW cm⁻² sr⁻¹)",
    rangemode="tozero",
    row=2, col=1,
)
fig.update_yaxes(
    title_text="NTL difference",
    row=3, col=1,
)

fig.update_layout(
    legend=dict(
        orientation="h",
        x=0, y=-.12,
        xanchor="left", yanchor="top",
        font=dict(size=16),
    ),
)
fig.update_annotations(font=dict(size=18, color=PLOT_TEXT_COLOR))

finish_figure(fig, "regional_g3_daily_stl_period_sensitivity")

## 5. GHSL-class baseline

Four-day spatial completeness and radiance are shown by GHSL class before the class profile panels. Monthly guides make timing comparable without treating missing blocks as zero.

In [ ]:
pre = four_day.loc[
    four_day["date_start"].ge(CHARACTERIZATION_START)
    & four_day["date_end"].le(CHARACTERIZATION_END)
].copy()

Each heatmap row is a settlement class; greener blocks have greater spatial completeness. The blank calendar remains informative.

In [ ]:
class_names = [
    f"{code} · {label}"
    for code, label in GHSL_CLASS_LABELS.items()
]

coverage = (
    pre.loc[pre.profile.isin(class_names)]
    .pivot(index="profile", columns="date", values="spatial_coverage_pct")
    .reindex(class_names)
)

fig = go.Figure(go.Heatmap(
    x=coverage.columns,
    y=coverage.index,
    z=coverage.values,
    colorscale=SC_GREEN_COLORSCALE,
    zmin=0, zmax=100,
    hoverongaps=False,
    colorbar=dict(
        title=dict(text="SC (%)", font=dict(size=18)),
        tickfont=dict(size=16),
        thickness=18,
    ),
    hovertemplate=(
        "%{y}<br>%{x|%d %b %Y}"
        "<br>Four-day SC: %{z:.1f}%<extra></extra>"
    ),
))

style_figure(fig, "", 1600, 300)
fig.update_layout(
    title=None,
    font=dict(size=17),
    margin=dict(l=230, r=90, t=25, b=65),
)

fig.update_xaxes(
    tickformat="%b %Y",
    dtick="M3",
    tick0="2012-01-01",
    tickfont=dict(size=15),
    showgrid=False,
)

month_starts = pd.date_range(
    coverage.columns.min().to_period("M").to_timestamp(),
    coverage.columns.max().to_period("M").to_timestamp(),
    freq="MS",
)

for month in month_starts:
    date = month.strftime("%Y-%m-%d")
    fig.add_shape(
        type="line",
        xref="x", yref="paper",
        x0=date, x1=date,
        y0=0, y1=1,
        line=dict(color="black", width=1),
        layer="above",
    )

fig.update_yaxes(
    autorange="reversed",
    tickfont=dict(size=16), title_text=None,
    showgrid=False,
)
finish_figure(fig, "07_four_day_availability")

Radiance is shown only where the four-day profile is observable. Compare this with the preceding completeness heatmap before interpreting apparent class changes.

In [ ]:
levels = (
    pre.loc[pre.profile.isin(class_names)]
    .pivot(index="profile", columns="date", values="raw_median_ntl")
    .reindex(class_names)
)
sc = coverage.reindex(index=levels.index, columns=levels.columns)

# Log colour display only; hover reports the original radiance.
display_values = np.log1p(
    np.clip(levels.values, 0, NTL_DISPLAY_MAX)
)
hover_values = np.stack([levels.values, sc.values], axis=-1)

radiance_ticks = [0, .2, .5, 1, 2, 5]

fig = go.Figure(go.Heatmap(
    x=levels.columns,
    y=levels.index,
    z=display_values,
    customdata=hover_values,
    colorscale="Inferno",
    zmin=0,
    zmax=np.log1p(NTL_DISPLAY_MAX),
    hoverongaps=False,
    colorbar=dict(
        title=dict(text="NTL", font=dict(size=18)),
        tickvals=np.log1p(radiance_ticks),
        ticktext=[str(value) for value in radiance_ticks],
        tickfont=dict(size=16),
        thickness=18,
    ),
    hovertemplate=(
        "%{y}<br>%{x|%d %b %Y}"
        "<br>RQ NTL: %{customdata[0]:.3f} nW cm⁻² sr⁻¹"
        "<br>SC: %{customdata[1]:.1f}%<extra></extra>"
    ),
))

style_figure(fig, "", 1600, 300)
fig.update_layout(
    title=None,
    font=dict(size=17),
    margin=dict(l=230, r=90, t=25, b=65),
)
fig.update_xaxes(
    tickformat="%b %Y",
    dtick="M3",
    tick0="2012-01-01",
    tickfont=dict(size=15),
    showgrid=False,
)

month_starts = pd.date_range(
    coverage.columns.min().to_period("M").to_timestamp(),
    coverage.columns.max().to_period("M").to_timestamp(),
    freq="MS",
)

for month in month_starts:
    date = month.strftime("%Y-%m-%d")
    fig.add_shape(
        type="line",
        xref="x", yref="paper",
        x0=date, x1=date,
        y0=0, y1=1,
        line=dict(color="black", width=1),
        layer="above",
    )

fig.update_yaxes(
    autorange="reversed",
    tickfont=dict(size=16), title_text=None,
    showgrid=False,
)
finish_figure(fig, "08_four_day_levels")

The class panels place daily source values behind the reliability-qualified four-day trajectory. Their shared dates support comparisons of timing rather than simple level ranking.

In [ ]:
first_month = pd.Timestamp(CHARACTERIZATION_START).to_period("M").to_timestamp()
month_starts = pd.date_range(first_month, CHARACTERIZATION_END, freq="MS")
half_year_ticks = pd.date_range(first_month, CHARACTERIZATION_END, freq="6MS")

RAW_DOT_COLOR = "#87919B"


def baseline_panels(names, colors, ncols, filename, show=True):
    nrows = int(np.ceil(len(names) / ncols))
    fig = make_subplots(
        rows=2*nrows, cols=ncols,
        row_heights=[.07, .43]*nrows,
        vertical_spacing=.09,
        horizontal_spacing=.055,
    )
    style_figure(fig, "", 1800 if ncols == 4 else 1600, 800)

    for i, (name, color) in enumerate(zip(names, colors)):
        col = i % ncols + 1
        sc_row = 2*(i // ncols) + 1
        ntl_row = sc_row + 1

        d = (
            daily.loc[daily.profile.eq(name)]
            .set_index("date")
            .reindex(calendar)
            .loc[CHARACTERIZATION_START:CHARACTERIZATION_END]
        )
        g = four_day.loc[
            four_day.profile.eq(name)
            & four_day.date_start.ge(CHARACTERIZATION_START)
            & four_day.date_end.le(CHARACTERIZATION_END)
        ].sort_values("date")

        fig.add_trace(go.Heatmap(
            x=g.date, y=["SC"], z=[g.spatial_coverage_pct],
            colorscale=SC_GREEN_COLORSCALE,
            zmin=0, zmax=100,
            showscale=(i == 0),
            colorbar=dict(
                title=dict(text="SC (%)", font=dict(size=12)),
                tickfont=dict(size=12), len=.1, x = 1, y=.99, thickness=15,
            ),
            hovertemplate=(
                "%{x|%d %b %Y}<br>Four-day SC: %{z:.1f}%"
                "<extra></extra>"
            ),
            hoverongaps=False,
        ), row=sc_row, col=col)

        # Gray daily observations sit behind the RQ line.
        fig.add_trace(go.Scatter(
            x=d.index, y=d.raw_uncapped,
            mode="markers",
            marker=dict(size=3.5, color=RAW_DOT_COLOR, opacity=.35),
            showlegend=False,
            hovertemplate=(
                "%{x|%d %b %Y}<br>Uncapped daily NTL: %{y:.4f}"
                "<extra></extra>"
            ),
        ), row=ntl_row, col=col)

        legend_id = "legend" if i == 0 else f"legend{i+1}"
        fig.add_trace(go.Scatter(
            x=g.date, y=g.raw_median_ntl,
            mode="lines",
            name=name,
            legend=legend_id,
            showlegend=True,
            line=dict(color=color, width=1.6, shape="hv"),
            connectgaps=False,
            customdata=g[["spatial_coverage_pct", "valid_pixels"]],
            hovertemplate=(
                "%{x|%d %b %Y}<br>Four-day RQ NTL: %{y:.4f}"
                "<br>SC: %{customdata[0]:.1f}%"
                "<br>Valid pixels: %{customdata[1]:.0f}"
                "<extra></extra>"
            ),
        ), row=ntl_row, col=col)

        # Class legend sits between its SC strip and NTL panel.
        axis_number = (ntl_row-1)*ncols + col
        suffix = "" if axis_number == 1 else str(axis_number)
        xdomain = fig.layout[f"xaxis{suffix}"].domain
        ydomain = fig.layout[f"yaxis{suffix}"].domain
        fig.update_layout(**{
            legend_id: dict(
                x=xdomain[0],
                y=ydomain[1] + .007,
                xanchor="left",
                yanchor="bottom",
                font=dict(size=16),
                bgcolor="rgba(0,0,0,0)",
                borderwidth=0,
            )
        })

        for row in (sc_row, ntl_row):
            fig.update_xaxes(
                range=[CHARACTERIZATION_START, CHARACTERIZATION_END],
                tickvals=half_year_ticks,
                tickformat="%b %Y",
                tickfont=dict(size=15),
                showticklabels=(row == ntl_row),
                title_text=None,
                showgrid=False,
                row=row, col=col,
            )

        fig.update_yaxes(
            showticklabels=False,
            showgrid=False,
            title_text=None,
            row=sc_row, col=col,
        )

        is_centre = name in profile_meta.loc[
            profile_meta.kind.eq("centre"), "profile"
        ].values

        if is_centre:
            values = np.concatenate([
                d.raw_uncapped.to_numpy(dtype=float),
                g.raw_median_ntl.to_numpy(dtype=float),
            ])
            values = values[np.isfinite(values)]
            ymax = (
                max(0.5, np.ceil(values.max()*1.05*2)/2)
                if values.size else 1.2
            )
        else:
            ymax = 5 if name.startswith("30 ·") else 1.2

        fig.update_yaxes(
            range=[0, ymax],
            tickmode="auto",
            nticks=5,
            tickfont=dict(size=15),
            title=dict(
                text="NTL (nW cm⁻² sr⁻¹)" if col == 1 else "",
                font=dict(size=17),
            ),
            row=ntl_row, col=col,
        )

    fig.update_layout(
        title=None,
        font=dict(size=16),
        margin=dict(l=95, r=115, t=45, b=65),
    )

    if show:
        finish_figure(fig, filename)
    return fig


# Seven GHSL classes; panel eight compares them on one axis.
class_names = [
    f"{code} · {label}"
    for code, label in GHSL_CLASS_LABELS.items()
]
class_colors = [
    GHSL_CLASS_COLORS[code]
    for code in GHSL_CLASS_LABELS
]

fig = baseline_panels(
    class_names, class_colors, 4,
    "03_ghsl_baseline", show=False,
)

coverage_rows = []

for code, name, color in zip(
    GHSL_CLASS_LABELS, class_names, class_colors
):
    d = (
        daily.loc[daily.profile.eq(name)]
        .set_index("date")
        .reindex(calendar)
        .loc[CHARACTERIZATION_START:CHARACTERIZATION_END]
    )
    g = four_day.loc[
        four_day.profile.eq(name)
        & four_day.date_start.ge(CHARACTERIZATION_START)
        & four_day.date_end.le(CHARACTERIZATION_END)
    ].sort_values("date")

    coverage_rows.append(
        g.set_index("date").spatial_coverage_pct.rename(str(code))
    )

    fig.add_trace(go.Scatter(
        x=d.index, y=d.raw_uncapped,
        mode="markers",
        marker=dict(size=3, color=RAW_DOT_COLOR, opacity=.13),
        showlegend=False,
        hoverinfo="skip",
    ), row=4, col=4)

    fig.add_trace(go.Scatter(
        x=g.date, y=g.raw_median_ntl,
        mode="lines",
        line=dict(color=color, width=1.6, shape="hv"),
        name=name,
        showlegend=False,
        connectgaps=False,
        hovertemplate=(
            f"{name}<br>"
            "%{x|%d %b %Y}<br>Four-day RQ NTL: %{y:.4f}"
            "<extra></extra>"
        ),
    ), row=4, col=4)

# Seven separate SC rows. Missing coverage remains missing.
coverage = pd.concat(coverage_rows, axis=1).sort_index()
fig.add_trace(go.Heatmap(
    x=coverage.index,
    y=coverage.columns.tolist(),
    z=coverage.T.values,
    colorscale=SC_GREEN_COLORSCALE,
    zmin=0, zmax=100,
    showscale=False,
    hoverongaps=False,
    hovertemplate=(
        "GHSL %{y}<br>%{x|%d %b %Y}"
        "<br>Four-day SC: %{z:.1f}%<extra></extra>"
    ),
), row=3, col=4)

# Match the date labels in the comparison panel.
for row in (3, 4):
    fig.update_xaxes(
        range=[CHARACTERIZATION_START, CHARACTERIZATION_END],
        tickvals=half_year_ticks,
        tickformat="%b %Y",
        tickfont=dict(size=15),
        showticklabels=(row == 4),
        title_text=None,
        showgrid=False,
        row=row, col=4,
    )

# Faint monthly guides on every NTL panel.
for ntl_row in (2, 4):
    for col in range(1, 5):
        for month in month_starts:
            fig.add_vline(
                x=month,
                line_color=(
                    "rgba(36,59,90,.15)"
                    if month.month == 1
                    else "rgba(36,59,90,.08)"
                ),
                line_width=1 if month.month == 1 else .7,
                layer="below",
                row=ntl_row, col=col,
            )

fig.update_yaxes(
    showticklabels=False,
    showgrid=False,
    title_text=None,
    row=3, col=4,
)
fig.update_yaxes(
    range=[0, 5],
    dtick=1,
    tickfont=dict(size=15),
    title_text=None,
    row=4, col=4,
)

finish_figure(fig, "03_ghsl_baseline")

The six centre panels apply the same observation grammar at named locations. Uncapped four-day levels are labelled separately from the regional P95-capped RQ series.

In [ ]:
centre_meta = profile_meta.loc[
    profile_meta.kind.eq("centre")
]

fig = baseline_panels(
    names=centre_meta.profile.tolist(),
    colors=centre_meta.color.tolist(),
    ncols=3,
    filename="04_six_centres",
    show=False,
)

for name in centre_meta.profile:
    g = four_day.loc[
        four_day.profile.eq(name)
        & four_day.date_start.ge(CHARACTERIZATION_START)
        & four_day.date_end.le(CHARACTERIZATION_END)
    ].sort_values("date")

    # Fresh-pixel four-day median, without the regional P95 cap.
    # Keep the existing spatial-completeness qualification.
    uncapped_4d = g.raw_uncapped.where(g.observed)

    for trace in fig.data:
        if trace.name == name:
            trace.y = uncapped_4d.to_numpy()
            trace.hovertemplate = (
                "%{x|%d %b %Y}"
                "<br>Uncapped four-day NTL: %{y:.2f}"
                "<br>SC: %{customdata[0]:.1f}%"
                "<br>Valid pixels: %{customdata[1]:.0f}"
                "<extra></extra>"
            )
            break

finish_figure(fig, "04_six_centres")

## 6. Municipal and 5×5 baseline comparisons

The existing profiles, annual-recurrence checks, and threshold sensitivity are retained. Their estimands differ with spatial support, so compare timing, coverage, and variability together.

In [ ]:
class_codes = [30, 23, 22]
class_profiles = [
    f"{code} · {GHSL_CLASS_LABELS[code]}"
    for code in class_codes
]
poi_profiles = [
    "Airport vicinity | 5×5",
    "Port vicinity | 5×5",
    "Plaza vicinity | 5×5",
]
scales = {
    "Regional GHSL": class_profiles,
    "Tacloban POI": poi_profiles,
}

pre = four_day.loc[
    four_day.date_start.ge(CHARACTERIZATION_START)
    & four_day.date_end.le(CHARACTERIZATION_END)
].copy()

rows = []
for scale, names in scales.items():
    for name in names:
        g = pre.loc[pre.profile.eq(name)].sort_values("date")
        reference = ntl0[fixed[name]]

        for threshold in (10., 50., 60.):
            retained = g.loc[
                g.spatial_coverage_pct.ge(threshold)
                & g.valid_pixels.gt(0)
            ]
            ntl = retained.rq_median.dropna()
            relative = retained.reference_pct.dropna()

            rows.append(dict(
                scale=scale,
                profile=name,
                sc_threshold_pct=threshold,
                fixed_pixels=len(fixed[name]),
                calendar_blocks=len(g),
                observed_blocks=len(ntl),
                retained_pct=100*len(ntl)/len(g),
                reference_median_ntl=(
                    np.median(reference) if len(reference) else np.nan
                ),
                observed_median_ntl=(
                    ntl.median() if len(ntl) else np.nan
                ),
                observed_p10_ntl=(
                    ntl.quantile(.10) if len(ntl) else np.nan
                ),
                observed_p90_ntl=(
                    ntl.quantile(.90) if len(ntl) else np.nan
                ),
                temporal_mad_ntl=(
                    (ntl-ntl.median()).abs().median()
                    if len(ntl) else np.nan
                ),
                temporal_mad_pct=(
                    (relative-relative.median()).abs().median()
                    if len(relative) else np.nan
                ),
                median_sc_pct=g.spatial_coverage_pct.median(),
            ))

baseline_metrics = pd.DataFrame(rows)

print("Primary characterization · SC ≥10%")
display(
    baseline_metrics.loc[
        baseline_metrics.sc_threshold_pct.eq(10),
        [
            "scale", "profile", "fixed_pixels",
            "observed_blocks", "retained_pct",
            "reference_median_ntl", "observed_median_ntl",
            "observed_p10_ntl", "observed_p90_ntl",
            "temporal_mad_ntl", "temporal_mad_pct",
        ],
    ].round(2)
)

The regional GHSL-class and Tacloban 5×5 panels share the same four-day calendar. Spatial completeness above each curve indicates when an apparent change can be assessed.

In [ ]:
calendar = pd.date_range(dates.min(), dates.max(), freq="D")

for scale, names in scales.items():
    fig = make_subplots(
        rows=2, cols=3,
        row_heights=[.09, .91],
        vertical_spacing=.08,
        horizontal_spacing=.06,
    )
    style_figure(fig, "", 1650, 630)

    for i, name in enumerate(names):
        col = i + 1
        color = (
            GHSL_CLASS_COLORS[class_codes[i]]
            if scale == "Regional GHSL"
            else ["#0072B2", "#E69F00", "#D55E00"][i]
        )

        d = (
            daily.loc[daily.profile.eq(name)]
            .set_index("date")
            .reindex(calendar)
            .loc[CHARACTERIZATION_START:CHARACTERIZATION_END]
        )
        g = pre.loc[pre.profile.eq(name)].sort_values("date")

        fig.add_trace(go.Heatmap(
            x=g.date, y=["SC"], z=[g.spatial_coverage_pct],
            colorscale=SC_GREEN_COLORSCALE,
            zmin=0, zmax=100,
            showscale=(i == 0),
            colorbar=dict(
                title=dict(text="4-day SC (%)", font=dict(size=16)),
                tickfont=dict(size=15), len=.22, y=.95,
            ),
            hoverongaps=False,
            hovertemplate=(
                "%{x|%d %b %Y}<br>SC: %{z:.1f}%<extra></extra>"
            ),
        ), row=1, col=col)

        fig.add_trace(go.Scatter(
            x=d.index, y=d.raw_uncapped,
            mode="markers",
            marker=dict(size=4, color=DNB_COLOR, opacity=.42),
            showlegend=False,
            hovertemplate=(
                "%{x|%d %b %Y}<br>Uncapped daily NTL: %{y:.3f}"
                "<extra></extra>"
            ),
        ), row=2, col=col)

        legend_id = "legend" if i == 0 else f"legend{i+1}"
        fig.add_trace(go.Scatter(
            x=g.date, y=g.raw_median_ntl,
            mode="lines+markers",
            name=name.replace(" | ", " · "),
            legend=legend_id,
            line=dict(color=color, width=2.5, shape="hv"),
            marker=dict(size=6),
            connectgaps=False,
            customdata=g[["spatial_coverage_pct", "valid_pixels"]],
            hovertemplate=(
                "%{x|%d %b %Y}<br>RQ NTL: %{y:.3f}"
                "<br>SC: %{customdata[0]:.1f}%"
                "<br>Valid pixels: %{customdata[1]:.0f}"
                "<extra></extra>"
            ),
        ), row=2, col=col)

        xdomain = fig.layout[f"xaxis{col+3}"].domain
        ydomain = fig.layout[f"yaxis{col+3}"].domain
        fig.update_layout(**{
            legend_id: dict(
                x=xdomain[0], y=ydomain[1]+.008,
                xanchor="left", yanchor="bottom",
                font=dict(size=16),
                bgcolor="rgba(0,0,0,0)",
            )
        })

        values = np.concatenate([
            d.raw_uncapped.to_numpy(dtype=float),
            g.raw_median_ntl.to_numpy(dtype=float),
        ])
        values = values[np.isfinite(values)]
        ymax = (
            5 if name.startswith("30 ·")
            else 1.2 if scale == "Regional GHSL"
            else max(.5, np.ceil(values.max()*1.05*2)/2)
                 if values.size else 1.2
        )

        for row in (1, 2):
            fig.update_xaxes(
                range=[CHARACTERIZATION_START, CHARACTERIZATION_END],
                tickformat="%b %y", dtick="M6",
                tickfont=dict(size=15),
                showticklabels=(row == 2),
                title_text=None, row=row, col=col,
            )
        fig.update_yaxes(
            showticklabels=False, showgrid=False,
            row=1, col=col,
        )
        fig.update_yaxes(
            range=[0, ymax], nticks=5,
            tickfont=dict(size=15),
            title_text=(
                "NTL (nW cm⁻² sr⁻¹)" if col == 1 else None
            ),
            row=2, col=col,
        )

    fig.update_layout(
        title=None,
        font=dict(size=16),
        margin=dict(l=90, r=100, t=35, b=65),
    )
    finish_figure(
        fig,
        "baseline_ghsl_timeseries"
        if scale == "Regional GHSL"
        else "baseline_poi_timeseries",
    )

Year-over-year overlays test whether similar calendar timing recurs in observed blocks. The short baseline does not establish annual seasonality.

In [ ]:
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=class_profiles + poi_profiles,
    horizontal_spacing=.06,
    vertical_spacing=.12,
)
recurrence_rows = []

for i, name in enumerate(class_profiles + poi_profiles):
    row, col = divmod(i, 3)
    row += 1
    col += 1

    g = pre.loc[pre.profile.eq(name)].copy()
    g = g.loc[
        ~((g.date.dt.month == 2) & (g.date.dt.day == 29))
    ]
    g["year"] = g.date.dt.year
    g["season_day"] = (
        g.date.dt.dayofyear
        - (
            g.date.dt.is_leap_year
            & g.date.dt.dayofyear.gt(60)
        ).astype(int)
    )

    for year, color in [(2012, DNB_COLOR), (2013, "#E69F00")]:
        yearly = g.loc[g.year.eq(year)]
        fig.add_trace(go.Scatter(
            x=yearly.season_day,
            y=yearly.raw_median_ntl,
            mode="lines+markers",
            name=str(year),
            legendgroup=str(year),
            showlegend=(i == 0),
            line=dict(color=color, width=1.8),
            marker=dict(size=5),
            connectgaps=False,
            customdata=yearly[["spatial_coverage_pct"]],
            hovertemplate=(
                f"{year} · day %{{x}}"
                "<br>RQ NTL: %{y:.3f}"
                "<br>SC: %{customdata[0]:.1f}%"
                "<extra></extra>"
            ),
        ), row=row, col=col)

    for threshold in (10., 50., 60.):
        observed = g.loc[
            g.spatial_coverage_pct.ge(threshold)
            & g.valid_pixels.gt(0)
            & g.rq_median.notna()
        ]

        a = observed.loc[observed.year.eq(2012), [
            "season_day", "rq_median"
        ]].sort_values("season_day")
        b = observed.loc[observed.year.eq(2013), [
            "season_day", "rq_median"
        ]].sort_values("season_day")

        matched = pd.merge_asof(
            a, b, on="season_day",
            direction="nearest", tolerance=2,
            suffixes=("_2012", "_2013"),
        ).dropna()

        correlation = (
            matched.rq_median_2012.corr(matched.rq_median_2013)
            if len(matched) >= 20
            and matched.rq_median_2012.std() > 0
            and matched.rq_median_2013.std() > 0
            else np.nan
        )
        recurrence_rows.append(dict(
            profile=name,
            sc_threshold_pct=threshold,
            matched_year_pairs=len(matched),
            year_to_year_correlation=correlation,
            median_absolute_difference=(
                (matched.rq_median_2012
                 - matched.rq_median_2013).abs().median()
                if len(matched) else np.nan
            ),
        ))

style_figure(fig, "", 1650, 900)
fig.update_layout(
    title=None,
    font=dict(size=16),
    margin=dict(l=80, r=40, t=55, b=105),
    legend=dict(
        orientation="h", x=0, y=-.12,
        font=dict(size=17),
    ),
)
fig.update_annotations(font=dict(size=16, color=PLOT_TEXT_COLOR))
fig.update_xaxes(
    tickvals=[1, 60, 121, 182, 244, 305],
    ticktext=["Jan", "Mar", "May", "Jul", "Sep", "Nov"],
    tickfont=dict(size=15),
    title_text=None,
)
fig.update_yaxes(tickfont=dict(size=15))
finish_figure(fig, "baseline_year_to_year_recurrence")

recurrence = pd.DataFrame(recurrence_rows)
display(recurrence.round(2))

Tightening the spatial-completeness gate changes both retained evidence and measured variability. Treat these together when choosing a baseline rule.

In [ ]:
fig = make_subplots(rows=1, cols=2, horizontal_spacing=.12)

for threshold, color in [
    (10., RQ_FOUR_DAY_COLOR),
    (50., "#56B4E9"),
    (60., "#D55E00"),
]:
    g = baseline_metrics.loc[
        baseline_metrics.sc_threshold_pct.eq(threshold)
    ]

    for col, value in [
        (1, "temporal_mad_pct"),
        (2, "retained_pct"),
    ]:
        fig.add_trace(go.Bar(
            x=g.profile.str.replace(" | 5×5", "", regex=False),
            y=g[value],
            name=f"SC ≥ {threshold:.0f}%",
            legendgroup=str(threshold),
            showlegend=(col == 1),
            marker_color=color,
            hovertemplate="%{x}<br>%{y:.1f}<extra></extra>",
        ), row=1, col=col)

style_figure(fig, "", 1650, 660)
fig.update_layout(
    title=None, barmode="group",
    font=dict(size=16),
    margin=dict(l=90, r=40, t=25, b=145),
    legend=dict(
        orientation="h", x=0, y=-.25,
        font=dict(size=17),
    ),
)
fig.update_xaxes(tickangle=-25, tickfont=dict(size=14))
fig.update_yaxes(
    title_text="Temporal MAD (percentage points)",
    row=1, col=1,
)
fig.update_yaxes(
    title_text="Four-day blocks retained (%)",
    range=[0, 100], row=1, col=2,
)
finish_figure(fig, "baseline_sc_sensitivity")

window_rows = []
for poi in ["Airport vicinity", "Port vicinity", "Plaza vicinity"]:
    for size in (1, 3, 5):
        name = f"{poi} | {size}×{size}"
        g = pre.loc[pre.profile.eq(name)]
        y = g.normalized_ntl_pct.dropna()
        window_rows.append(dict(
            poi=poi,
            window=f"{size}×{size}",
            fixed_pixels=len(fixed[name]),
            observed_blocks=len(y),
            retained_pct=100*len(y)/len(g),
            temporal_mad_pct=(
                (y-y.median()).abs().median()
                if len(y) else np.nan
            ),
        ))

window_sensitivity = pd.DataFrame(window_rows)
display(window_sensitivity.round(2))

The next existing landmark figure uses a plotting helper and summary table from the earlier baseline notebook. This setup restores those definitions without changing the figure cell.

In [ ]:
def profile_grid(names,title,filename,value="raw_median_ntl",pre_only=False):
    ncols=min(3,len(names)); ngroups=int(np.ceil(len(names)/ncols))
    fig=make_subplots(rows=2*ngroups,cols=ncols,row_heights=[.07,.43]*ngroups,
                      horizontal_spacing=.065,vertical_spacing=.07)
    for i,name in enumerate(names):
        col=i%ncols+1; sr=2*(i//ncols)+1; pr=sr+1
        g=four_day[four_day.profile.eq(name)].sort_values("date")
        if pre_only: g=g[g.date_start.ge(CHARACTERIZATION_START)&g.date_end.le(CHARACTERIZATION_END)]
        color=profile_meta.set_index("profile").loc[name,"color"]
        fig.add_trace(go.Heatmap(x=g.date,y=["SC"],z=[g.spatial_coverage_pct],colorscale=SC_GREEN_COLORSCALE,
            zmin=0,zmax=100,showscale=(i==0),colorbar=dict(title="SC (%)",len=.10,y=.98,thickness=13),
            hovertemplate="%{x|%d %b %Y}<br>SC %{z:.1f}%<extra></extra>",hoverongaps=False),row=sr,col=col)
        if value=="raw_median_ntl":
            d=daily[daily.profile.eq(name)].set_index("date").reindex(calendar)
            if pre_only: d=d.loc[CHARACTERIZATION_START:CHARACTERIZATION_END]
            fig.add_trace(go.Scatter(x=d.index,y=d.raw_median_ntl,mode="markers",marker=dict(size=2,color=color),
                opacity=.22,showlegend=False,hoverinfo="skip"),row=pr,col=col)
        fig.add_trace(go.Scatter(x=g.date,y=g[value],mode="lines+markers",connectgaps=False,
            line=dict(color=color,width=2.8,shape="hv"),marker_size=4,showlegend=False,
            customdata=g[["spatial_coverage_pct","valid_pixels"]],
            hovertemplate="%{x|%d %b %Y}<br>NTL %{y:.2f}<br>SC %{customdata[0]:.1f}%<br>Pixels %{customdata[1]}<extra></extra>"),row=pr,col=col)
        label=name.replace(" | "," · ")
        axis=(pr-1)*ncols+col; xref="x" if axis==1 else f"x{axis}"; yref="y" if axis==1 else f"y{axis}"
        fig.add_annotation(x=.02,y=.98,xref=xref+" domain",yref=yref+" domain",text=f"<b>{label}</b>",
                           showarrow=False,xanchor="left",yanchor="top",font=dict(size=15,color=color))
        if value=="normalized_ntl_pct": fig.add_hline(y=100,line_dash="dot",line_color=BASELINE_COLOR,row=pr,col=col)
        for row in (sr,pr):
            if not pre_only: add_haiyan_marker(fig,row,col)
            fig.update_xaxes(range=[CHARACTERIZATION_START,CHARACTERIZATION_END if pre_only else dates.max()],
                tickformat="%b<br>%Y",nticks=5,showticklabels=(row==pr),row=row,col=col)
        fig.update_yaxes(showticklabels=False,showgrid=False,row=sr,col=col)
        fig.update_yaxes(title_text=("% 60-day reference" if value=="normalized_ntl_pct" else "Median NTL") if col==1 else None,
                         rangemode="tozero",row=pr,col=col)
    for empty in range(len(names),ncols*ngroups):
        col=empty%ncols+1; sr=2*(empty//ncols)+1
        for row in (sr,sr+1):
            fig.update_xaxes(visible=False,row=row,col=col)
            fig.update_yaxes(visible=False,row=row,col=col)
    style_figure(fig,title,1400,460*ngroups)
    finish_figure(fig,filename)


pre=four_day[four_day.date_start.ge(CHARACTERIZATION_START)&four_day.date_end.le(CHARACTERIZATION_END)].copy()
rows=[]
for name,g in pre.groupby("profile",sort=False):
    reference_values=ntl0[fixed[name]]
    q25,level,q75=np.quantile(reference_values,[.25,.5,.75]) if len(reference_values) else [np.nan]*3
    series=g.normalized_ntl_pct.dropna()
    rows.append(dict(profile=name,baseline_q25=q25,baseline_median_ntl=level,baseline_q75=q75,
        temporal_mad_pct=(series-series.median()).abs().median() if len(series) else np.nan,
        observed_blocks=len(series),calendar_blocks=len(g),observed_share=len(series)/len(g),
        median_sc_pct=g.spatial_coverage_pct.median() if g.spatial_coverage_pct.notna().any() else np.nan,
        characterization_median_ntl=g.raw_median_ntl.median() if g.raw_median_ntl.notna().any() else np.nan))
summary=pd.DataFrame(rows).merge(profile_meta,on="profile")
summary["fixed_support_share"]=summary.fixed_pixels/summary.base_pixels.replace(0,np.nan)
summary["characterization_start"],summary["characterization_end"]=CHARACTERIZATION_START,CHARACTERIZATION_END
summary["reference_start"],summary["reference_end"]=BASELINE_START,PRE_EVENT_END
summary["sc_threshold_pct"],summary["p95"]=SPATIAL_COMPLETENESS_PCT,RQ_CLIP_PERCENTILE

The 5×5 landmark panels and threshold plots ask how baseline variation changes with support size and the SC rule; they do not imply a disaster effect.

In [ ]:
landmark_names=profile_meta.loc[profile_meta.kind.eq("landmark")&profile_meta.extent.eq("5×5"),"profile"].tolist()
profile_grid(landmark_names,"Landmark hypotheses · 5×5 G3 neighbourhoods","09_landmark_profiles",pre_only=True)
sensitivity_rows=[]
for threshold in (10.,50.,60.):
    for name,g in pre.groupby("profile",sort=False):
        y=g.loc[g.spatial_coverage_pct.ge(threshold)&g.valid_pixels.gt(0),"reference_pct"].dropna()
        sensitivity_rows.append(dict(profile=name,sc_threshold_pct=threshold,observed_blocks=len(y),
            retained_pct=100*len(y)/len(g),temporal_mad_pct=(y-y.median()).abs().median() if len(y) else np.nan))
sensitivity=pd.DataFrame(sensitivity_rows).merge(profile_meta,on="profile")
fig=make_subplots(rows=1,cols=2,subplot_titles=["Variability changes with the SC gate","Evidence retained"],horizontal_spacing=.12)
for threshold,color in [(10.,RQ_FOUR_DAY_COLOR),(50.,"#56B4E9"),(60.,"#D55E00")]:
    g=sensitivity[sensitivity.profile.isin(landmark_names)&sensitivity.sc_threshold_pct.eq(threshold)]
    short=g.profile.str.replace(" | 5×5","",regex=False)
    for col,value in [(1,"temporal_mad_pct"),(2,"retained_pct")]:
        fig.add_trace(go.Scatter(x=short,y=g[value],mode="markers",name=f"SC ≥ {threshold:.0f}%",legendgroup=str(threshold),
            showlegend=col==1,marker=dict(size=11,color=color),customdata=g[["observed_blocks"]],
            hovertemplate="%{x}<br>%{y:.2f}<br>Observed blocks %{customdata[0]}<extra></extra>"),row=1,col=col)
style_figure(fig,"Landmark screening · threshold sensitivity",1500,700)
fig.update_yaxes(title_text="Temporal MAD (pp)",row=1,col=1)
fig.update_yaxes(title_text="Retained blocks (%)",row=1,col=2)
finish_figure(fig,"10_threshold_sensitivity")
window_table=summary[summary.kind.eq("landmark")].copy()
window_table["site"]=window_table.profile.str.split(" | ",regex=False).str[0]
fig=px.scatter(window_table,x="site",y="temporal_mad_pct",color="extent",symbol="extent",
    color_discrete_map={"1×1":"#56B4E9","3×3":"#E69F00","5×5":"#009E73"},
    hover_data=["observed_blocks","fixed_pixels","fixed_support_share"])
fig.update_traces(marker_size=12)
style_figure(fig,"Spatial-window sensitivity · observed baseline variability")
finish_figure(fig,"11_window_sensitivity")
display(window_table[["profile","fixed_pixels","observed_blocks","median_sc_pct","temporal_mad_pct"]].sort_values("temporal_mad_pct"))


## 7. Pre-event model and decomposition checks

These retained diagnostics describe only the pre-Haiyan baseline. Notebook 07 develops the post-Haiyan STL scenarios and impact comparisons.

### Pre-event decomposition

STL decomposes an interpolated **model input** into trend, an assumed 28-day component, and residual. Solid segments coincide with observed RQ blocks; dotted segments cross supplied inputs.

In [ ]:
# Regional Samar–Leyte G3: reliability-qualified NTL and exploratory STL
from statsmodels.tsa.seasonal import STL

regional_daily = pd.read_csv(
    cache_dir / "daily.csv",
    usecols=[
        "date", "profile", "raw_uncapped",
        "valid_pixels", "spatial_coverage_pct",
    ],
    parse_dates=["date"],
)
regional_daily = regional_daily.loc[
    regional_daily["profile"].eq("Regional G3")
    & regional_daily["date"].between(
        CHARACTERIZATION_START, CHARACTERIZATION_END
    )
].sort_values("date")

regional_blocks = pd.read_csv(
    cache_dir / "four_day.csv",
    usecols=[
        "date", "profile", "rq_median",
        "valid_pixels", "spatial_coverage_pct",
    ],
    parse_dates=["date"],
)
regional_blocks = regional_blocks.loc[
    regional_blocks["profile"].eq("Regional G3")
].copy()

# Include only four-day blocks wholly inside the pre-Haiyan period.
regional_blocks["date_start"] = (
    regional_blocks["date"] - pd.Timedelta(days=1.5)
)
regional_blocks["date_end"] = (
    regional_blocks["date"] + pd.Timedelta(days=1.5)
)
regional_blocks = regional_blocks.loc[
    regional_blocks["date_start"].ge(CHARACTERIZATION_START)
    & regional_blocks["date_end"].le(CHARACTERIZATION_END)
].sort_values("date")

calendar = pd.DatetimeIndex(regional_blocks["date"])
sc = pd.Series(
    regional_blocks["spatial_coverage_pct"].to_numpy(),
    index=calendar,
)
qualified = (
    regional_blocks["spatial_coverage_pct"].ge(
        SPATIAL_COMPLETENESS_PCT
    )
    & regional_blocks["valid_pixels"].gt(0)
)
rq = pd.Series(
    regional_blocks["rq_median"].where(qualified).to_numpy(),
    index=calendar,
)

if rq.notna().sum() < 14:
    raise ValueError("Too few observed regional G3 blocks for 28-day STL.")

# Missing blocks receive values solely to make a regular STL input.
# The observed RQ series remains unchanged.
stl_input = (
    rq.interpolate(method="time", limit_area="inside")
    .ffill()
    .bfill()
)
observed = rq.notna()
missing = ~observed
gap_groups = missing.ne(missing.shift(fill_value=False)).cumsum()
longest_gap = int(missing.groupby(gap_groups).sum().max())

display(pd.DataFrame([{
    "profile": "Regional G3",
    "calendar_blocks": len(calendar),
    "observed_RQ_blocks": int(observed.sum()),
    "blocks_supplied_to_STL": int(missing.sum()),
    "supplied_pct": round(100 * missing.mean(), 1),
    "longest_gap_blocks": longest_gap,
    "SC_threshold_pct": SPATIAL_COMPLETENESS_PCT,
}]))

# Seven four-day blocks represent a 28-day diagnostic component.
stl = STL(np.log1p(stl_input.to_numpy()), period=7, robust=True).fit()
trend = pd.Series(np.expm1(stl.trend), index=calendar)
seasonal = pd.Series(stl.seasonal, index=calendar)
residual = pd.Series(stl.resid, index=calendar)

fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    row_heights=[.25, .375, .225, .15],
    vertical_spacing=.055,
    subplot_titles=[
        "Observed nighttime lights",
        "Trend",
        "Seasonality · 28-day diagnostic",
        "Residual",
    ],
)

fig.add_trace(go.Scatter(
    x=regional_daily["date"],
    y=regional_daily["raw_uncapped"],
    mode="markers",
    marker=dict(color=DNB_COLOR, size=4, opacity=.38),
    name="Daily DNB-BRDF NTL",
    customdata=regional_daily[
        ["valid_pixels", "spatial_coverage_pct"]
    ].to_numpy(),
    hovertemplate=(
        "%{x|%d %b %Y}<br>Daily DNB-BRDF NTL: %{y:.2f}"
        "<br>Valid pixels: %{customdata[0]:.0f}"
        "<br>SC: %{customdata[1]:.0f}%<extra></extra>"
    ),
), row=1, col=1)

# The solid observed series covers this dashed input wherever RQ data exist.
# Visible dashed spans therefore identify blocks supplied to STL.
fig.add_trace(go.Scatter(
    x=calendar,
    y=stl_input,
    mode="lines",
    line=dict(color=BASELINE_COLOR, width=1.6, dash="dot"),
    name="STL input",
    hovertemplate="%{x|%d %b %Y}<br>STL input: %{y:.2f}<extra></extra>",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=calendar,
    y=rq,
    mode="lines+markers",
    connectgaps=False,
    line=dict(color=RQ_FOUR_DAY_COLOR, width=2.6),
    marker=dict(color=RQ_FOUR_DAY_COLOR, size=4),
    name="RQ NTL (4D)",
    customdata=sc.to_numpy().reshape(-1, 1),
    hovertemplate=(
        "%{x|%d %b %Y}<br>RQ NTL (4D): %{y:.2f}"
        "<br>SC: %{customdata[0]:.0f}%<extra></extra>"
    ),
), row=1, col=1)

for row, values, color, label in [
    (2, trend, "#174A7E", "Trend"),
    (3, seasonal, "#E69F00", "Seasonality"),
    (4, residual, "#6C7882", "Residual"),
]:
    # Dashed portions show the decomposition through supplied input blocks.
    fig.add_trace(go.Scatter(
        x=calendar,
        y=values,
        mode="lines",
        line=dict(color=color, width=1.7, dash="dot"),
        opacity=.7,
        showlegend=False,
        hovertemplate=(
            f"%{{x|%d %b %Y}}<br>{label}: %{{y:.3f}}"
            "<br>STL model input<extra></extra>"
        ),
    ), row=row, col=1)

    fig.add_trace(go.Scatter(
        x=calendar,
        y=values.where(observed),
        mode="lines",
        connectgaps=False,
        line=dict(color=color, width=2.7),
        showlegend=False,
        hovertemplate=(
            f"%{{x|%d %b %Y}}<br>{label}: %{{y:.3f}}"
            "<br>Observed RQ block<extra></extra>"
        ),
    ), row=row, col=1)

for row in (3, 4):
    fig.add_hline(
        y=0, line_color="#AAB5C2", line_width=1,
        row=row, col=1,
    )

fig.update_yaxes(
    title_text="NTL (nW cm⁻² sr⁻¹)", rangemode="tozero",
    row=1, col=1,
)
fig.update_yaxes(
    title_text="Trend NTL<br>(nW cm⁻² sr⁻¹)", rangemode="tozero",
    row=2, col=1,
)
fig.update_yaxes(
    title_text="Seasonality<br>(log1p NTL)",
    row=3, col=1,
)
fig.update_yaxes(
    title_text="Residual<br>(log1p NTL)",
    row=4, col=1,
)

style_figure(fig, "", 1600, 980)

for row in range(1, 5):
    fig.update_xaxes(
        range=[display_start, CHARACTERIZATION_END],
        showgrid=False,
        tickmode="array",
        tickvals=quarter_ticks,
        ticktext=[d.strftime("%b %Y") for d in quarter_ticks],
        tickfont=dict(size=14),
        showticklabels=(row == 4),
        row=row, col=1,
    )
    fig.update_yaxes(tickfont=dict(size=14), row=row, col=1)

    for month in month_starts:
        fig.add_vline(
            x=month,
            line_color=(
                "rgba(36,59,90,.20)"
                if month.month == 1
                else "rgba(36,59,90,.10)"
            ),
            line_width=1.2 if month.month == 1 else .8,
            layer="below",
            row=row, col=1,
        )

fig.update_layout(
    legend=dict(
        orientation="h", x=0, y=-.08,
        xanchor="left", yanchor="top",
        font=dict(size=16),
    ),
)
fig.update_annotations(font=dict(size=17, color=PLOT_TEXT_COLOR))
finish_figure(fig, "regional_g3_observed_and_stl")

Changing the STL trend span tests how much the estimated baseline curve depends on smoothing. The 28-day component remains fixed across these fits.

In [ ]:
# Regional G3 · sensitivity to the STL trend span
# Four-day blocks; seven blocks form the 28-day seasonal component.

trend_windows = {
    9:  ("≈1 month · 36 d",  "#009E73"),
    15: ("≈2 months · 60 d", "#243B5A"),
    23: ("≈3 months · 92 d", "#E69F00"),
    31: ("≈4 months · 124 d", "#CC79A7"),
}
reference_blocks = 15
observed_blocks = rq.notna()

trend_curves = {}
for blocks in trend_windows:
    fit = STL(
        np.log1p(stl_input.to_numpy()),
        period=7,
        seasonal=7,
        trend=blocks,
        low_pass=9,
        robust=True,
    ).fit()
    trend_curves[blocks] = pd.Series(
        np.expm1(fit.trend),
        index=calendar,
    )

reference = trend_curves[reference_blocks]

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[.72, .28],
    vertical_spacing=.10,
    subplot_titles=[
        "Observed NTL and STL trend",
        "Difference from the 60-day setting",
    ],
)

fig.add_trace(go.Scatter(
    x=calendar,
    y=rq,
    mode="lines",
    connectgaps=False,
    line=dict(color=RQ_FOUR_DAY_COLOR, width=1.2),
    opacity=.4,
    name="RQ NTL (4D)",
    hovertemplate="%{x|%d %b %Y}<br>RQ NTL: %{y:.2f}<extra></extra>",
), row=1, col=1)

for blocks, (label, color) in trend_windows.items():
    curve = trend_curves[blocks]
    width = 3 if blocks == reference_blocks else 2

    # The dotted curve remains visible only across missing RQ blocks.
    fig.add_trace(go.Scatter(
        x=calendar,
        y=curve,
        mode="lines",
        line=dict(color=color, width=width, dash="dot"),
        opacity=.65,
        showlegend=False,
        hovertemplate=(
            f"%{{x|%d %b %Y}}<br>{label}"
            "<br>Trend: %{y:.2f}<br>Supplied STL input<extra></extra>"
        ),
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=calendar,
        y=curve.where(observed_blocks),
        mode="lines",
        connectgaps=False,
        line=dict(color=color, width=width),
        name=label,
        hovertemplate=(
            f"%{{x|%d %b %Y}}<br>{label}"
            "<br>Trend: %{y:.2f}<extra></extra>"
        ),
    ), row=1, col=1)

    if blocks != reference_blocks:
        difference = curve - reference

        fig.add_trace(go.Scatter(
            x=calendar,
            y=difference,
            mode="lines",
            line=dict(color=color, width=2, dash="dot"),
            opacity=.65,
            showlegend=False,
            hovertemplate=(
                f"%{{x|%d %b %Y}}<br>{label}"
                "<br>Difference: %{y:+.2f}<extra></extra>"
            ),
        ), row=2, col=1)

        fig.add_trace(go.Scatter(
            x=calendar,
            y=difference.where(observed_blocks),
            mode="lines",
            connectgaps=False,
            line=dict(color=color, width=2),
            showlegend=False,
            hovertemplate=(
                f"%{{x|%d %b %Y}}<br>{label}"
                "<br>Difference: %{y:+.2f}<extra></extra>"
            ),
        ), row=2, col=1)

fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode="lines",
    line=dict(color="#7A8794", width=2, dash="dot"),
    name="Dotted · missing-block fit",
), row=1, col=1)

fig.add_hline(
    y=0, row=2, col=1,
    line_color=PLOT_TEXT_COLOR,
    line_width=1,
)

first_month = calendar.min().to_period("M").to_timestamp()
month_starts = pd.date_range(
    first_month, calendar.max(), freq="MS"
)
quarter_ticks = pd.date_range(
    first_month, calendar.max(), freq="3MS"
)

style_figure(fig, "", 1600, 800)

for row in (1, 2):
    fig.update_xaxes(
        range=[CHARACTERIZATION_START, CHARACTERIZATION_END],
        tickmode="array",
        tickvals=quarter_ticks,
        ticktext=[d.strftime("%b %Y") for d in quarter_ticks],
        tickfont=dict(size=14),
        showgrid=False,
        row=row, col=1,
    )
    fig.update_yaxes(tickfont=dict(size=14), row=row, col=1)

    for month in month_starts:
        fig.add_vline(
            x=month,
            row=row, col=1,
            line_color=PLOT_GRID_COLOR,
            line_width=.8,
            opacity=.55,
            layer="below",
        )

fig.update_yaxes(
    title_text="NTL (nW cm⁻² sr⁻¹)",
    rangemode="tozero",
    row=1, col=1,
)
fig.update_yaxes(
    title_text="Difference in NTL",
    row=2, col=1,
)

fig.update_layout(
    legend=dict(
        orientation="h",
        x=0, y=-.12,
        xanchor="left", yanchor="top",
        font=dict(size=16),
    ),
)
fig.update_annotations(font=dict(size=18, color=PLOT_TEXT_COLOR))
finish_figure(fig, "regional_g3_stl_trend_span_sensitivity")

Class-specific decomposition is exploratory. Dotted spans reflect interpolated inputs and should not be mistaken for observed nighttime lights.

In [ ]:
ghsl_pre = four_day.loc[
    four_day["date_start"].ge(CHARACTERIZATION_START)
    & four_day["date_end"].le(CHARACTERIZATION_END)
].copy()

calendar = pd.DatetimeIndex(
    sorted(ghsl_pre["date"].dropna().unique())
)

In [ ]:
# Exploratory GHSL-class STL · pre-Haiyan four-day calendar
# Seven four-day blocks represent the 28-day component.

from statsmodels.tsa.seasonal import STL

STL_PERIOD_BLOCKS = 7

class_groups = {
    1: (11, 12, 13, 21),
    2: (22, 23, 30),
}

calendar = pd.DatetimeIndex(sorted(pre.date.dropna().unique()))
stl_frames = []
audit_rows = []

for code in (11, 12, 13, 21, 22, 23, 30):
    name = f"{code} · {GHSL_CLASS_LABELS[code]}"
    g = ghsl_pre.loc[
        ghsl_pre["profile"].eq(name)
    ].sort_values("date")

    observed = (
        g.set_index("date")["raw_median_ntl"]
        .reindex(calendar)
    )
    sc = (
        g.set_index("date")["spatial_coverage_pct"]
        .reindex(calendar)
    )

    if observed.notna().sum() < 2:
        audit_rows.append(dict(
            ghsl=code,
            observed_blocks=int(observed.notna().sum()),
            interpolated_blocks=np.nan,
            status="Too few observations",
        ))
        continue

    missing = observed.isna()
    gap_id = missing.ne(missing.shift(fill_value=False)).cumsum()
    longest_gap = max(
        (len(run) for _, run in missing.groupby(gap_id) if run.all()),
        default=0,
    )

    # Interior gaps are interpolated; edge gaps carry the nearest
    # observed value. These values supply STL, not the RQ series.
    filled = (
        observed.interpolate(method="time", limit_area="inside")
        .ffill()
        .bfill()
    )

    result = STL(
        np.log1p(filled.to_numpy()),
        period=STL_PERIOD_BLOCKS,
        robust=True,
    ).fit()

    stl_frames.append(pd.DataFrame({
        "date": calendar,
        "ghsl": code,
        "rq_ntl": observed.to_numpy(),
        "sc_pct": sc.to_numpy(),
        "observed_input": ~missing.to_numpy(),
        "trend": np.asarray(result.trend),
        "trend_ntl": np.expm1(np.asarray(result.trend)),
        "seasonal": np.asarray(result.seasonal),
        "residual": np.asarray(result.resid),
    }))

    audit_rows.append(dict(
        ghsl=code,
        observed_blocks=int((~missing).sum()),
        interpolated_blocks=int(missing.sum()),
        interpolated_pct=100 * missing.mean(),
        longest_gap_blocks=longest_gap,
        status="STL fitted",
    ))

stl_audit = pd.DataFrame(audit_rows)
display(stl_audit.round(1))

if not stl_frames:
    print("No GHSL class has enough observed blocks for STL.")
else:
    stl_classes = pd.concat(stl_frames, ignore_index=True)

    fig = make_subplots(
        rows=4, cols=2,
        shared_xaxes=True,
        row_heights=[.15, .43, .26, .16],
        horizontal_spacing=.05,
        vertical_spacing=.055,
        subplot_titles=[
            "Lower-density classes", "Urban classes",
            "", "",
            "", "",
            "", "",
        ],
    )

    for col, codes in class_groups.items():
        available_codes = [
            code for code in codes
            if stl_classes.ghsl.eq(code).any()
        ]

        # A separate SC strip for every class, on the same time axis.
        sc_grid = np.vstack([
            stl_classes.loc[
                stl_classes.ghsl.eq(code), "sc_pct"
            ].to_numpy()
            for code in available_codes
        ])

        fig.add_trace(go.Heatmap(
            x=calendar,
            y=[str(code) for code in available_codes],
            z=sc_grid,
            colorscale=SC_GREEN_COLORSCALE,
            zmin=0,
            zmax=100,
            hoverongaps=False,
            showscale=(col == 2),
            colorbar=dict(
                title=dict(text="SC (%)", font=dict(size=15)),
                tickfont=dict(size=14),
                tickvals=[0, 50, 100],
                thickness=16,
                len=.13,
                y=.95,
                x=1.01,
            ),
            hovertemplate=(
                "GHSL %{y}<br>%{x|%d %b %Y}"
                "<br>Four-day SC: %{z:.1f}%<extra></extra>"
            ),
        ), row=1, col=col)

        for code in available_codes:
            g = stl_classes.loc[
                stl_classes.ghsl.eq(code)
            ].sort_values("date")
            color = GHSL_CLASS_COLORS[code]
            class_name = f"{code} · {GHSL_CLASS_LABELS[code]}"

            # Thin steps show the observed four-day RQ NTL.
            fig.add_trace(go.Scatter(
                x=g.date,
                y=g.rq_ntl,
                mode="lines",
                connectgaps=False,
                line=dict(color=color, width=1.3, shape="hv"),
                opacity=.45,
                showlegend=False,
                legendgroup=str(code),
                customdata=g[["sc_pct"]].to_numpy(),
                hovertemplate=(
                    f"{class_name}<br>%{{x|%d %b %Y}}"
                    "<br>RQ NTL (4D): %{y:.2f}"
                    "<br>SC: %{customdata[0]:.0f}%"
                    "<extra></extra>"
                ),
            ), row=2, col=col)

            # Keep the fitted trend continuous across supplied blocks.
            # Convert it back to NTL units to compare with RQ NTL.
            fig.add_trace(go.Scatter(
                x=g.date,
                y=g.trend_ntl,
                mode="lines",
                line=dict(color=color, width=2.8),
                name=class_name,
                legendgroup=str(code),
                hovertemplate=(
                    f"{class_name}<br>%{{x|%d %b %Y}}"
                    "<br>STL trend: %{y:.2f} NTL"
                    "<extra></extra>"
                ),
            ), row=2, col=col)

            fig.add_trace(go.Scatter(
                x=g.date,
                y=g.seasonal,
                mode="lines",
                line=dict(color=color, width=1.7),
                opacity=.85,
                showlegend=False,
                legendgroup=str(code),
                hovertemplate=(
                    f"{class_name}<br>%{{x|%d %b %Y}}"
                    "<br>28-day component: %{y:.3f}"
                    "<extra></extra>"
                ),
            ), row=3, col=col)

            # Residuals from supplied input are not shown as evidence.
            fig.add_trace(go.Scatter(
                x=g.date,
                y=g.residual.where(g.observed_input),
                mode="markers",
                marker=dict(color=color, size=4, opacity=.65),
                showlegend=False,
                legendgroup=str(code),
                hovertemplate=(
                    f"{class_name}<br>%{{x|%d %b %Y}}"
                    "<br>Observed-block residual: %{y:.3f}"
                    "<extra></extra>"
                ),
            ), row=4, col=col)

        for row in (3, 4):
            fig.add_hline(
                y=0,
                line_color=PLOT_GRID_COLOR,
                line_width=1,
                row=row, col=col,
            )

    fig.update_yaxes(
        title_text="GHSL class",
        autorange="reversed",
        tickfont=dict(size=14),
        row=1, col=1,
    )
    fig.update_yaxes(
        autorange="reversed",
        tickfont=dict(size=14),
        row=1, col=2,
    )

    for col in (1, 2):
        fig.update_yaxes(
            title_text="NTL (nW cm⁻² sr⁻¹)" if col == 1 else None,
            rangemode="tozero",
            tickfont=dict(size=14),
            row=2, col=col,
        )
        fig.update_yaxes(
            title_text="28-day component<br>(log1p NTL)"
            if col == 1 else None,
            tickfont=dict(size=14),
            row=3, col=col,
        )
        fig.update_yaxes(
            title_text="Residual<br>(log1p NTL)"
            if col == 1 else None,
            tickfont=dict(size=14),
            row=4, col=col,
        )

    first_month = calendar.min().to_period("M").to_timestamp()
    month_starts = pd.date_range(
        first_month, calendar.max(), freq="MS"
    )
    quarter_ticks = pd.date_range(
        first_month, calendar.max(), freq="3MS"
    )

    style_figure(fig, "", 1600, 900)

    for row in (1, 2, 3, 4):
        for col in (1, 2):
            fig.update_xaxes(
                range=[calendar.min(), calendar.max()],
                tickmode="array",
                tickvals=quarter_ticks,
                ticktext=[d.strftime("%b %Y") for d in quarter_ticks],
                tickfont=dict(size=14),
                showticklabels=(row == 4),
                showgrid=False,
                row=row, col=col,
            )

            for month in month_starts:
                fig.add_vline(
                    x=month,
                    row=row, col=col,
                    line_color=PLOT_GRID_COLOR,
                    line_width=.8,
                    opacity=.5,
                    layer="above" if row == 1 else "below",
                )

    fig.update_layout(
        legend=dict(
            orientation="h",
            x=0, y=-.08,
            xanchor="left", yanchor="top",
            font=dict(size=15),
        ),
    )
    fig.update_annotations(font=dict(size=17, color=PLOT_TEXT_COLOR))

    finish_figure(fig, "ghsl_four_day_stl_full_calendar")

In [ ]:
# Exploratory GHSL-class STL using the full pre-Haiyan four-day calendar
# Missing blocks are interpolated for this diagnostic, never set to zero.

from statsmodels.tsa.seasonal import STL

STL_PERIOD_BLOCKS = 7  # 28-day component; not annual seasonality
class_groups = {
    1: (11, 12, 13, 21),
    2: (22, 23, 30),
}

calendar = pd.DatetimeIndex(sorted(pre.date.dropna().unique()))
stl_frames = []
audit_rows = []

for code in (11, 12, 13, 21, 22, 23, 30):
    name = f"{code} · {GHSL_CLASS_LABELS[code]}"
    g = pre[pre.profile.eq(name)].sort_values("date")

    observed = (
        g.set_index("date").raw_median_ntl
        .reindex(calendar)
    )

    if observed.notna().sum() < 2:
        audit_rows.append(dict(
            ghsl=code, observed_blocks=int(observed.notna().sum()),
            interpolated_blocks=np.nan, status="Too few observations",
        ))
        continue

    missing = observed.isna()
    gap_id = missing.ne(missing.shift(fill_value=False)).cumsum()
    longest_gap = max(
        (len(run) for _, run in missing.groupby(gap_id) if run.all()),
        default=0,
    )

    # Interior gaps: linear interpolation in calendar time.
    # Edge gaps: carry the nearest observed level to complete the STL input.
    filled = (
        observed.interpolate(method="time", limit_area="inside")
        .ffill()
        .bfill()
    )

    result = STL(
        np.log1p(filled),
        period=STL_PERIOD_BLOCKS,
        robust=True,
    ).fit()

    stl_frames.append(pd.DataFrame({
        "date": calendar,
        "ghsl": code,
        "observed_input": ~missing.to_numpy(),
        "trend": np.asarray(result.trend),
        "seasonal": np.asarray(result.seasonal),
        "residual": np.asarray(result.resid),
    }))

    audit_rows.append(dict(
        ghsl=code,
        observed_blocks=int((~missing).sum()),
        interpolated_blocks=int(missing.sum()),
        interpolated_pct=100 * missing.mean(),
        longest_gap_blocks=longest_gap,
        status="STL fitted",
    ))

stl_audit = pd.DataFrame(audit_rows)
display(stl_audit.round(1))

if not stl_frames:
    print("No GHSL class has enough observed blocks for STL.")
else:
    stl_classes = pd.concat(stl_frames, ignore_index=True)

    fig = make_subplots(
        rows=3, cols=2,
        shared_xaxes=True,
        row_heights=[.50, .30, .20],
        horizontal_spacing=.10,
        vertical_spacing=.07,
        subplot_titles=[
            "Classes 11–21 · trend", "Classes 22–30 · trend",
            "Classes 11–21 · 28-day component",
            "Classes 22–30 · 28-day component",
            "Classes 11–21 · residual",
            "Classes 22–30 · residual",
        ],
    )

    for col, codes in class_groups.items():
        for code in codes:
            g = stl_classes[
                stl_classes.ghsl.eq(code)
            ].sort_values("date")
            if g.empty:
                continue

            color = GHSL_CLASS_COLORS[code]
            for row, component in enumerate(
                ("trend", "seasonal", "residual"), start=1
            ):
                # Dotted line completes the component through filled blocks.
                fig.add_trace(
                    go.Scatter(
                        x=g.date,
                        y=g[component],
                        mode="lines",
                        line=dict(color=color, width=1.8, dash="dot"),
                        opacity=.65,
                        showlegend=False,
                        hovertemplate=(
                            f"GHSL {code}<br>%{{x|%d %b %Y}}"
                            f"<br>{component.capitalize()} %{{y:.3f}}"
                            "<br>STL includes interpolated input"
                            "<extra></extra>"
                        ),
                    ),
                    row=row, col=col,
                )

                # Solid line appears only at originally observed blocks.
                observed_component = g[component].where(g.observed_input)
                fig.add_trace(
                    go.Scatter(
                        x=g.date,
                        y=observed_component,
                        mode="lines",
                        line=dict(color=color, width=2.8),
                        connectgaps=False,
                        name=f"{code} · {GHSL_CLASS_LABELS[code]}",
                        legendgroup=str(code),
                        showlegend=(row == 1),
                        hovertemplate=(
                            f"GHSL {code}<br>%{{x|%d %b %Y}}"
                            f"<br>{component.capitalize()} %{{y:.3f}}"
                            "<br>Originally observed RQ block"
                            "<extra></extra>"
                        ),
                    ),
                    row=row, col=col,
                )

        for row in (2, 3):
            fig.add_hline(
                y=0, line_color=PLOT_GRID_COLOR,
                line_width=1, row=row, col=col,
            )

    for row, label in [
        (1, "Trend · log1p NTL"),
        (2, "28-day component · log1p"),
        (3, "Residual · log1p"),
    ]:
        fig.update_yaxes(
            title_text=label, tickfont=dict(size=14),
            row=row, col=1,
        )
        fig.update_yaxes(
            tickfont=dict(size=14),
            row=row, col=2,
        )

    for col in (1, 2):
        fig.update_xaxes(
            range=[calendar.min(), calendar.max()],
            tickformat="%b %Y",
            dtick="M3",
            tickfont=dict(size=14),
            row=3, col=col,
        )

    # Legend key for the dotted portions.
    fig.add_trace(
        go.Scatter(
            x=[None], y=[None], mode="lines",
            line=dict(color=BASELINE_COLOR, dash="dot", width=2),
            name="Interpolated-input portion",
            showlegend=True,
        ),
        row=1, col=1,
    )

    style_figure(fig, "", 1850, 1280)
    fig.update_layout(
        margin=dict(l=105, r=45, t=85, b=170),
        legend=dict(
            orientation="h",
            x=0, y=-.10,
            xanchor="left", yanchor="top",
            font=dict(size=16),
        ),
    )
    fig.update_annotations(font=dict(size=17, color=PLOT_TEXT_COLOR))
    finish_figure(fig, "ghsl_four_day_stl_full_calendar")

### Preliminary baseline-model check
For one profile, compare a constant median, a linear trend, and trend plus one annual harmonic on observed four-day RQ levels:

$\log(1+L_t)=\beta_0+\beta_1 t+\beta_2\sin(2\pi t)+\beta_3\cos(2\pi t)+\epsilon_t$.

Reserve the last 90 calendar days; exclude any block crossing the split. Report MAE in radiance units. The pixel support was fixed retrospectively using the original 60-day reference, so this is a **conditional descriptive comparison**, not prospective forecast validation. Less than two full pre-event annual cycles prevents a strong annual-seasonality claim. Start with the simplest model and inspect the coverage strip.


The held-out model comparison evaluates a baseline rule on observed four-day blocks; it is a conditional check on the fixed reference support.

In [ ]:
MODEL_PROFILE="30 · Urban centre"  # Also accepts an established centre or landmark profile.
g=pre[pre.profile.eq(MODEL_PROFILE)].copy()
cutoff=CHARACTERIZATION_END-pd.Timedelta(days=90)
train=g[g.date_end.lt(cutoff)].dropna(subset=["raw_median_ntl"])
test=g[g.date_start.ge(cutoff)].dropna(subset=["raw_median_ntl"])
def design(index,harmonic=False):
    t=np.asarray((pd.DatetimeIndex(index)-CHARACTERIZATION_START).days)/365.25
    cols=[np.ones(len(t)),t]
    if harmonic: cols.extend([np.sin(2*np.pi*t),np.cos(2*np.pi*t)])
    return np.column_stack(cols)
model_scores=pd.DataFrame()
if len(train)>=30 and len(test)>=10:
    predictions={"Constant median":np.repeat(train.raw_median_ntl.median(),len(test))}
    for name,harmonic in [("Linear trend",False),("Trend + annual harmonic",True)]:
        if harmonic and (train.date.max()-train.date.min()).days<365: continue
        beta=np.linalg.lstsq(design(train.date,harmonic),np.log1p(train.raw_median_ntl),rcond=None)[0]
        predictions[name]=np.maximum(0,np.expm1(design(test.date,harmonic)@beta))
    model_scores=pd.DataFrame([dict(model=name,heldout_mae=np.mean(abs(test.raw_median_ntl.values-y)),
        train_blocks=len(train),test_blocks=len(test),profile=MODEL_PROFILE) for name,y in predictions.items()]).sort_values("heldout_mae")
    fig=make_subplots(rows=2,cols=1,shared_xaxes=True,row_heights=[.10,.90],vertical_spacing=.05)
    fig.add_trace(go.Heatmap(x=g.date,y=["SC"],z=[g.spatial_coverage_pct],colorscale=SC_GREEN_COLORSCALE,
        zmin=0,zmax=100,colorbar=dict(title="SC (%)",len=.16,y=.97)),row=1,col=1)
    fig.add_trace(go.Scatter(x=g.date,y=g.raw_median_ntl,mode="lines+markers",connectgaps=False,
        line=dict(color=RQ_FOUR_DAY_COLOR,width=2.5,shape="hv"),marker_size=4,name="Observed RQ · four-day"),row=2,col=1)
    for (name,y),color in zip(predictions.items(),[BASELINE_COLOR,DNB_COLOR,"#E69F00"]):
        fig.add_trace(go.Scatter(x=test.date,y=y,mode="markers",marker=dict(color=color,size=7),name=name),row=2,col=1)
    fig.add_vline(x=cutoff.to_pydatetime(),line_dash="dot",line_color=BASELINE_COLOR,row=2,col=1)
    fig.update_yaxes(showticklabels=False,showgrid=False,row=1,col=1)
    fig.update_yaxes(title_text="Median NTL (nW cm⁻² sr⁻¹)",row=2,col=1)
    style_figure(fig,f"Baseline models · {MODEL_PROFILE}")
    finish_figure(fig,"12_model_comparison"); display(model_scores)
else: print("Too few observed train/test blocks for this profile; select another MODEL_PROFILE.")


### Optional uninterrupted-segment STL and exports
STL is restricted to the longest consecutive run of qualified daily RQ observations; no interpolation and no post-disaster dates are used. At least 56 observed days are required for a preliminary 7-day component. Annual STL is deferred because the extended baseline spans fewer than two annual cycles. This diagnostic does not establish venue seasonality.


The optional uninterrupted-segment STL is a separate diagnostic. The same cell exports the notebook’s baseline tables and processing settings.

In [ ]:
try:
    from statsmodels.tsa.seasonal import STL
except ImportError:
    STL=None
s=daily[daily.profile.eq(MODEL_PROFILE)].set_index("date").raw_median_ntl.reindex(pd.date_range(CHARACTERIZATION_START,CHARACTERIZATION_END))
segments=[x for _,x in s.groupby((s.isna()!=s.isna().shift()).cumsum()) if x.notna().all()]
segment=max(segments,key=len) if segments else s.iloc[:0]
if STL is not None and len(segment)>=56:
    result=STL(np.log1p(segment),period=7,robust=True).fit()
    fig=make_subplots(rows=3,cols=1,shared_xaxes=True,subplot_titles=["Trend","7-day component","Residual"])
    for row,values,color in [(1,result.trend,RQ_FOUR_DAY_COLOR),(2,result.seasonal,DNB_COLOR),(3,result.resid,BASELINE_COLOR)]:
        fig.add_trace(go.Scatter(x=segment.index,y=values,mode="lines",line=dict(color=color),showlegend=False),row=row,col=1)
    style_figure(fig,"STL on a fully observed segment · log1p units",1280,900)
    finish_figure(fig,"13_stl_diagnostic")
else: print(f"STL skipped: longest observed run {len(segment)} days; requires ≥56 days and statsmodels.")

for name,table in [("daily_profiles",daily),("four_day_profiles",four_day),("baseline_summary",summary),
                   ("threshold_sensitivity",sensitivity),("profile_support",profile_meta),("model_comparison",model_scores)]:
    table.to_csv(TABLE_DIR/f"{name}.csv",index=False)
settings={"reference":[str(BASELINE_START.date()),str(PRE_EVENT_END.date())],
          "characterization":[str(CHARACTERIZATION_START.date()),str(CHARACTERIZATION_END.date())],
          "sc_threshold_pct":SPATIAL_COMPLETENESS_PCT,"p95":RQ_CLIP_PERCENTILE,"cache_key":cache_key,
          "ghsl":str(GHSL_PATH),"stores":[str(BASELINE_STORE),str(EXISTING_STORE)],
          "source_conventions":"Notebooks 02–04; G3 cap applied over regional support"}
(TABLE_DIR/"analysis_settings.json").write_text(json.dumps(settings,indent=2))
print("Figures:",FIGURE_DIR,"\nTables:",TABLE_DIR)


### Interpretation guide

The accompanying **NOTEBOOK_06_STYLE_GUIDE.md** records exact palette values, thresholds, plotting hierarchy, the original cells they came from, and the deliberate baseline-analysis extensions. Use it for future edits instead of substituting generic Plotly defaults.

- Inspect the support map and SC strips before curve shape. Ten-percent support is permissive.
- Compare 10/50/60% gates and 1×1/3×3/5×5 windows before shortlisting locations.
- Separate spatial heterogeneity, temporal variability and uncertainty. No plotted IQR or MAD is labelled a confidence interval.
- Validate landmark positions, historical use, observing geometry and event calendars before assigning causal or seasonal explanations.
- Regional G3, native-class and location profiles have different supports. Their metadata preserves that distinction.

### Evidence behind the preliminary characterization


| Evidence | Consequence here |
|---|---|
| [Román et al. (2018), NASA’s Black Marble nighttime lights product suite](https://www.sciencedirect.com/science/article/am/pii/S003442571830110X) | Use corrected DNB-BRDF and retain quality information; at-sensor and gap-filled signals are not interchangeable with fresh retrievals. |
| [NASA Black Marble User Guide](https://viirsland.gsfc.nasa.gov/PDF/BlackMarbleUserGuide_v1.2_20210421.pdf) | Preserve MQF and distinguish direct observations from gap filling. |
| [Román & Stokes (2015), Holidays in lights](https://doi.org/10.1002/2014EF000285) | Cultural schedules can affect lighting; this motivates testing, not assuming, analogous behaviour in Tacloban. |
| [GHSL settlement classification](https://human-settlement.emergency.copernicus.eu/ghs_smod2023.php) | Compare settlement morphology using native categorical classes. A 2015 mask is a temporal approximation for this baseline. |
| [Chakraborty & Stokes (2023)](https://arxiv.org/abs/2306.08501) | Expected lighting behaviour varies across cities; examine local baseline residuals rather than imposing one shared curve. |
| [STL documentation and original-method reference](https://www.statsmodels.org/stable/generated/statsmodels.tsa.seasonal.STL.html) | Decomposition is a diagnostic; regular spacing and sufficient temporal support matter. |

No source establishes that the particular airport/port/road pixels here are stable or the proposed plaza/arena pixels are seasonal. Those remain data-driven and location-verified questions.


## 8. Selected Tacloban pixels

The final section locates reference-qualified 1×1 pixels, then follows their four-day baseline and observability. The last figure extends into the post-Haiyan year while keeping each pre-Haiyan median fixed.

In [ ]:
# 1. Select distinct, observable pixels and show their spatial context

site_specs = [
    ("Airport", 125.02645, 11.22747, "#0072B2", "diamond"),
    ("Port", 124.99865, 11.24905, "#E69F00", "square"),
    ("Main-road vicinity", 125.00300, 11.24200, "#009E73", "cross"),
    ("BLISS Sagkahan residential", 125.00264, 11.21767, "#CC79A7", "triangle-up"),
    ("Kassel City residential", 124.98314, 11.23742, "#7B5EA7", "triangle-down"),
    ("Rizal Plaza", 125.00614, 11.24208, "#D55E00", "star"),
    ("Astrodome", 125.00502, 11.22165, "#56B4E9", "x"),
    ("Robinsons Place", 125.00778, 11.20752, "#B33D6A", "hexagon"),
]

In [ ]:
# Select distinct, reference-qualified pixels for the named sites
# and nearby GHSL comparison pixels.

map_bounds = (124.91, 11.13, 125.10, 11.33)

eligible = np.flatnonzero(reference_ok)
flat = land_pixels[eligible]
iy, ix = np.unravel_index(flat, land.shape)

candidate = pd.DataFrame({
    "ix": ix,
    "iy": iy,
    "x": a2.x.values[ix],
    "y": a2.y.values[iy],
    "ghsl": ghsl_vector[eligible].astype(int),
    "reference_ntl": ntl0[eligible],
})
candidate = candidate.loc[
    candidate.x.between(map_bounds[0], map_bounds[2])
    & candidate.y.between(map_bounds[1], map_bounds[3])
].copy()

if candidate.empty:
    raise ValueError("No reference-qualified land pixels in the POI map bounds.")

selected_sites = []
used_pixels = set()

for name, lon, lat, color, symbol in site_specs:
    available = candidate.loc[~candidate.index.isin(used_pixels)]
    distance = 111.2 * np.hypot(
        (available.x - lon) * np.cos(np.deg2rad(lat)),
        available.y - lat,
    )
    chosen_id = distance.idxmin()
    shift_km = float(distance.loc[chosen_id])

    if shift_km > 1.5:
        raise ValueError(
            f"{name}: nearest eligible pixel is {shift_km:.2f} km away."
        )

    p = available.loc[chosen_id]
    used_pixels.add(chosen_id)
    selected_sites.append(dict(
        site=name,
        x=float(p.x), y=float(p.y),
        ix=int(p.ix), iy=int(p.iy),
        ghsl=int(p.ghsl),
        reference_ntl=float(p.reference_ntl),
        shift_km=shift_km,
        color=color, symbol=symbol,
    ))

comparison_symbols = {
    23: "diamond", 22: "square", 21: "triangle-up",
    13: "circle", 12: "triangle-down", 11: "star",
}

for code, symbol in comparison_symbols.items():
    available = candidate.loc[
        candidate.ghsl.eq(code)
        & ~candidate.index.isin(used_pixels)
    ]
    if available.empty:
        print(f"No eligible GHSL {code} pixel within the map bounds.")
        continue

    distance = 111.2 * np.hypot(
        (available.x - 125.0015) * np.cos(np.deg2rad(11.2434)),
        available.y - 11.2434,
    )
    chosen_id = distance.idxmin()
    p = available.loc[chosen_id]
    used_pixels.add(chosen_id)

    selected_sites.append(dict(
        site=f"Pixel {code}",
        x=float(p.x), y=float(p.y),
        ix=int(p.ix), iy=int(p.iy),
        ghsl=int(p.ghsl),
        reference_ntl=float(p.reference_ntl),
        shift_km=np.nan,
        color=GHSL_CLASS_COLORS[code],
        symbol=symbol,
    ))

site_df = pd.DataFrame(selected_sites)

# The later 4×4 POI figures use the eight named sites only.
poi_sites = site_df.loc[
    site_df.site.isin([spec[0] for spec in site_specs])
].copy()
poi_colors = poi_sites.color.tolist()
poi_symbols = poi_sites.symbol.tolist()

# Fixed reference-qualified support for the spatial-window checks.
poi_fixed = {}
for site in site_df.itertuples():
    for size in (1, 3, 5):
        radius = size // 2
        window = np.zeros(land.shape, dtype=bool)
        window[
            max(0, site.iy-radius):site.iy+radius+1,
            max(0, site.ix-radius):site.ix+radius+1,
        ] = True
        poi_fixed[(site.site, size)] = np.flatnonzero(
            window.ravel()[land_pixels] & reference_ok
        )

display(
    site_df[
        ["site", "ghsl", "reference_ntl", "shift_km", "x", "y"]
    ].round(3)
)


Pixel outlines and labels mark the same selected cells on GHSL settlement and reference-NTL maps. The named sites are snapped to eligible observed pixels.

In [ ]:
# Selected 1×1 pixels on GHSL and pre-Haiyan reference NTL maps.
# Requires site_df and the display grids from the earlier map cell.

map_bounds = (124.91, 11.13, 125.10, 11.33)
xi = np.flatnonzero(
    (a2.x.values >= map_bounds[0]) & (a2.x.values <= map_bounds[2])
)
yi = np.flatnonzero(
    (a2.y.values >= map_bounds[1]) & (a2.y.values <= map_bounds[3])
)
dx = float(abs(a2.x.values[1] - a2.x.values[0]))
dy = float(abs(a2.y.values[1] - a2.y.values[0]))

fig = make_subplots(
    rows=1, cols=2,
    horizontal_spacing=.14,
    subplot_titles=[
        "Settlement classes (GHSL)",
        "Pre-Haiyan nighttime lights (60-day median)",
    ])

for col, grid in ((1, ghsl_grid), (2, reference_grid)):
    # Show water on the NTL panel without treating missing land NTL as water.
    if col == 2:
        fig.add_trace(go.Heatmap(
            x=a2.x.values[xi],
            y=a2.y.values[yi],
            z=water_grid[np.ix_(yi, xi)],
            colorscale=[[0, "#B8D9EF"], [1, "#B8D9EF"]],
            showscale=False,
            hoverinfo="skip",
        ), row=1, col=col)

    fig.add_trace(go.Heatmap(
        x=a2.x.values[xi],
        y=a2.y.values[yi],
        z=grid[np.ix_(yi, xi)],
        colorscale=ghsl_scale if col == 1 else "Inferno",
        zmin=-.5 if col == 1 else NTL_DISPLAY_MIN,
        zmax=len(map_classes) - .5 if col == 1 else NTL_DISPLAY_MAX,
        zsmooth=False,
        hoverongaps=False,
        colorbar=dict(
            x=.445 if col == 1 else 1.01,
            len=.72,
            thickness=17,
            title=dict(
                text="GHSL" if col == 1 else "NTL<br>(nW cm⁻² sr⁻¹)",
                font=dict(size=16),
            ),
            tickfont=dict(size=14),
            tickvals=list(range(len(map_classes))) if col == 1 else None,
            ticktext=[str(code) for code, _, _ in map_classes]
                     if col == 1 else None,
        ),
        hovertemplate=(
            "GHSL %{z:.0f}<extra></extra>" if col == 1
            else "%{y:.4f}°N, %{x:.4f}°E"
                 "<br>NTL %{z:.2f}<extra></extra>"
        ),
    ), row=1, col=col)

    fig.add_trace(go.Scatter(
        x=outline_x, y=outline_y,
        mode="lines",
        line=dict(color="rgba(36,59,90,.45)", width=.7),
        hoverinfo="skip",
        showlegend=False,
    ), row=1, col=col)

    fig.update_xaxes(
        range=[map_bounds[0], map_bounds[2]],
        tickformat=".2f", ticksuffix="°E",
        showgrid=False, row=1, col=col,
    )
    fig.update_yaxes(
        range=[map_bounds[1], map_bounds[3]],
        tickformat=".2f", ticksuffix="°N",
        showgrid=False,
        scaleanchor="x" if col == 1 else "x2",
        scaleratio=1 / np.cos(np.deg2rad(11.23)),
        row=1, col=col,
    )

style_figure(fig, "", 1850, 880)
fig.update_annotations(font=dict(size=21, color=PLOT_TEXT_COLOR))

# Road shapefile: use ROADS_PATH if set, otherwise find it under DATA_DIR.
road_path = globals().get("ROADS_PATH")
if road_path is None:
    road_files = sorted(DATA_DIR.glob("**/*[Rr]oad*.shp"))
    road_path = road_files[0] if road_files else None
if road_path is None:
    raise FileNotFoundError("Set ROADS_PATH to the road shapefile.")

roads_local = (
    gpd.read_file(road_path)
    .to_crs(a2.rio.crs)
    .cx[map_bounds[0]:map_bounds[2], map_bounds[1]:map_bounds[3]]
)

road_x, road_y = [], []
for geometry in roads_local.geometry.dropna():
    parts = geometry.geoms if hasattr(geometry, "geoms") else [geometry]
    for part in parts:
        if hasattr(part, "xy"):
            xx, yy = part.xy
            road_x.extend([*xx, None])
            road_y.extend([*yy, None])

for col in (1, 2):
    fig.add_trace(go.Scatter(
        x=road_x, y=road_y,
        mode="lines",
        line=dict(
            color="rgba(40,55,70,.55)" if col == 1
                  else "rgba(240,245,250,.65)",
            width=.9,
        ),
        hoverinfo="skip",
        showlegend=False,
    ), row=1, col=col)

    fig.update_xaxes(
        tickmode="array",
        tickvals=[124.95, 125.00, 125.05],
        ticktext=["124.95°E", "125.00°E", "125.05°E"],
        tickfont=dict(size=15),
        showgrid=False,
        row=1, col=col,
    )
    fig.update_yaxes(
        tickmode="array",
        tickvals=[11.20, 11.25, 11.30],
        ticktext=["11.20°N", "11.25°N", "11.30°N"],
        tickfont=dict(size=15),
        showgrid=False,
        row=1, col=col,
    )

# (panel, label x, label y, anchor, text, text color)
label_positions = {
    "Pixel 23": (1, 124.925, 11.305, "left",  "Pixel 23", PLOT_TEXT_COLOR),
    "Pixel 21": (1, 124.925, 11.265, "left",  "Pixel 21", PLOT_TEXT_COLOR),
    "Pixel 13": (1, 124.925, 11.225, "left",  "Pixel 13", PLOT_TEXT_COLOR),
    "Pixel 22": (1, 125.083, 11.295, "right", "Pixel 22", PLOT_TEXT_COLOR),
    "Pixel 11": (1, 125.083, 11.250, "right", "Pixel 11", PLOT_TEXT_COLOR),
    "Pixel 12": (1, 125.083, 11.205, "right", "Pixel 12", PLOT_TEXT_COLOR),

    "Port": (2, 124.925, 11.290, "left", "Port", "white"),
    "Kassel City residential": (
        2, 124.925, 11.250, "left", "Kassel City residential", "white"
    ),
    "BLISS Sagkahan residential": (
        2, 124.925, 11.210, "left", "BLISS Sagkahan residential", "white"
    ),
    "Main-road vicinity": (
        2, 125.025, 11.270, "left", "Main-road vicinity", PLOT_TEXT_COLOR
    ),
    "Rizal Plaza": (
        2, 125.030, 11.249, "left", "Rizal Plaza", PLOT_TEXT_COLOR
    ),
    "Airport": (
        2, 125.083, 11.245, "right", "Airport", PLOT_TEXT_COLOR
    ),
    "Astrodome": (
        2, 125.083, 11.220, "right", "Astrodome", PLOT_TEXT_COLOR
    ),
    "Robinsons Place": (
        2, 125.083, 11.195, "right", "Robinsons Place", PLOT_TEXT_COLOR
    ),
}

for p in site_df.itertuples():
    # Outline the selected 1×1 pixel on BOTH maps.
    for col in (1, 2):
        for edge_color, edge_width in (("white", 4), ("black", 2)):
            fig.add_shape(
                type="rect",
                x0=p.x - dx / 2,
                x1=p.x + dx / 2,
                y0=p.y - dy / 2,
                y1=p.y + dy / 2,
                line=dict(color=edge_color, width=edge_width),
                fillcolor="rgba(0,0,0,0)",
                layer="above",
                row=1, col=col,
            )

    # Label each pixel on just one panel.
    col, label_x, label_y, anchor, label, color = label_positions[p.site]
    xref = "x" if col == 1 else "x2"
    yref = "y" if col == 1 else "y2"

    fig.add_annotation(
        x=p.x, y=p.y,
        xref=xref, yref=yref,
        ax=label_x, ay=label_y,
        axref=xref, ayref=yref,
        text=label,
        xanchor=anchor,
        showarrow=True,
        arrowhead=2,
        arrowsize=.7,
        arrowwidth=1,
        arrowcolor=color,
        bgcolor="rgba(0,0,0,0)",
        borderwidth=0,
        font=dict(
            size=16,
            color=color,
            shadow=(
                "0 0 3px #07152C, 0 0 6px #07152C"
                if color == "white"
                else "0 0 3px white, 0 0 6px white"
            ),
        ),
    )

for trace in fig.data:
    if isinstance(trace, go.Heatmap) and trace.showscale:
        trace.colorbar.tickfont = dict(size=16)
        trace.colorbar.title.font = dict(size=18)

fig.update_layout(
    margin=dict(l=80, r=135, t=75, b=55),
    showlegend=False,
)
finish_figure(fig, "poi_ghsl_ntl_pixel_boxes")

In [ ]:
# Extract daily and four-day RQ NTL for those selected pixels and windows.
# Run once; the following plot cells can then be rerun without re-extraction.

poi_daily_rows = []
poi_four_rows = []

first_block = int(
    (CHARACTERIZATION_START - EVENT_DATE).days // AGGREGATION_DAYS
)
last_block = int(
    (PRE_EVENT_END - EVENT_DATE).days // AGGREGATION_DAYS
)

for block in tqdm(
    range(first_block, last_block + 1),
    desc="Selected POI windows",
):
    start = EVENT_DATE + pd.Timedelta(
        days=block * AGGREGATION_DAYS
    )
    end = start + pd.Timedelta(days=AGGREGATION_DAYS - 1)
    midpoint = start + pd.Timedelta(
        days=(AGGREGATION_DAYS - 1) / 2
    )
    full_block = (
        start >= CHARACTERIZATION_START
        and end <= PRE_EVENT_END
    )
    indices = np.flatnonzero(day_blocks == block)

    if len(indices):
        fresh, capped, _ = read_days(indices)
        composite = median(capped, axis=0)
    else:
        composite = np.full(
            len(land_pixels), np.nan, dtype=np.float32
        )

    for site in site_df.site:
        for size in (1, 3, 5):
            support = poi_fixed[(site, size)]
            n_fixed = len(support)

            # Daily values, including days in an incomplete edge block.
            if len(indices):
                for j, date_index in enumerate(indices):
                    date = dates[date_index]
                    if not CHARACTERIZATION_START <= date <= PRE_EVENT_END:
                        continue

                    values = capped[j, support]
                    n_valid = int(np.isfinite(values).sum())
                    sc_pct = (
                        100 * n_valid / n_fixed
                        if n_fixed else np.nan
                    )
                    observed = (
                        n_valid > 0
                        and sc_pct >= SPATIAL_COMPLETENESS_PCT
                    )

                    poi_daily_rows.append(dict(
                        site=site,
                        window=size,
                        date=date,
                        raw_uncapped=median(fresh[j, support]),
                        ntl=median(values) if observed else np.nan,
                        sc_pct=sc_pct,
                        valid_pixels=n_valid,
                        n_fixed=n_fixed,
                        observed=observed,
                    ))

            # Four-day results use complete pre-Haiyan blocks only.
            if not full_block:
                continue

            values = composite[support]
            valid = np.isfinite(values)
            n_valid = int(valid.sum())
            sc_pct = (
                100 * n_valid / n_fixed
                if n_fixed else np.nan
            )
            observed = (
                n_valid > 0
                and sc_pct >= SPATIAL_COMPLETENESS_PCT
            )
            level = median(values) if observed else np.nan
            reference_sum = (
                ntl0[support][valid].sum()
                if n_valid else np.nan
            )

            poi_four_rows.append(dict(
                site=site,
                window=size,
                date=midpoint,
                date_start=start,
                date_end=end,
                ntl=level,
                rq_median=level,
                median_ntl=level,
                q25=(
                    np.quantile(values[valid], .25)
                    if observed else np.nan
                ),
                q75=(
                    np.quantile(values[valid], .75)
                    if observed else np.nan
                ),
                sc_pct=sc_pct,
                n_valid=n_valid,
                valid_pixels=n_valid,
                n_fixed=n_fixed,
                observed=observed,
                reference_pct=(
                    100 * values[valid].sum() / reference_sum
                    if observed and reference_sum > 0
                    else np.nan
                ),
            ))

poi_daily = pd.DataFrame(poi_daily_rows)
poi_four_day = pd.DataFrame(poi_four_rows)
poi_box = poi_four_day

print(
    f"{len(site_df)} selected pixels; "
    f"{len(poi_four_day)} four-day window records."
)

The selected 1×1 time series reveal baseline variation and gaps at individual pixels; their magnitudes differ, so compare timing and observability as well as level.

In [ ]:
# Four-day RQ NTL at selected 1×1 pixels

plot_size = 1
urban = site_df[site_df.ghsl.eq(30)].copy()
other = site_df[site_df.ghsl.ne(30)].copy()

missing_sites = set(site_df.site) - set(poi_box.site)
if missing_sites:
    raise ValueError(
        "Rerun the four-day extraction cell for: "
        + ", ".join(sorted(missing_sites))
    )

pixel_symbols = {
    23: "diamond", 22: "square", 21: "triangle-up",
    13: "circle", 12: "triangle-down", 11: "star",
}

def add_group_key(fig, group, columns, x0, width, heading):
    fig.add_annotation(
        x=x0, y=-.09, xref="paper", yref="paper",
        text=heading, showarrow=False, xanchor="left",
        font=dict(size=18, color=PLOT_TEXT_COLOR),
    )
    for i, p in enumerate(group.itertuples()):
        key_row, key_col = divmod(i, columns)
        x = x0 + key_col * width / columns
        y = -.16 - key_row * .065
        color = (
            p.color if p.ghsl == 30
            else GHSL_CLASS_COLORS[int(p.ghsl)]
        )
        fig.add_annotation(
            x=x, y=y, xref="paper", yref="paper",
            text="●", showarrow=False, xanchor="left",
            font=dict(size=24, color=color),
        )
        fig.add_annotation(
            x=x + .022, y=y, xref="paper", yref="paper",
            text=p.site, showarrow=False, xanchor="left",
            font=dict(size=16, color=PLOT_TEXT_COLOR),
        )

fig = make_subplots(
    rows=1, cols=2,
    horizontal_spacing=.10,
    subplot_titles=[
        "Class 30 · named locations",
        "Other GHSL pixels · log1p display",
    ],
)

for col, group in [(1, urban), (2, other)]:
    for p in group.itertuples():
        g = poi_box[
            poi_box.site.eq(p.site)
            & poi_box.window.eq(plot_size)
        ].sort_values("date")

        color = (
            p.color if col == 1
            else GHSL_CLASS_COLORS[int(p.ghsl)]
        )
        symbol = (
            p.symbol if col == 1
            else pixel_symbols.get(int(p.ghsl), "circle")
        )
        display_y = (
            g.median_ntl if col == 1
            else np.log1p(g.median_ntl)
        )

        fig.add_trace(
            go.Scatter(
                x=g.date,
                y=display_y,
                mode="lines+markers",
                line=dict(color=color, width=2.1),
                marker=dict(
                    color=color, symbol=symbol, size=4,
                    line=dict(color=PLOT_TEXT_COLOR, width=.35),
                ),
                opacity=.85,
                connectgaps=False,
                name=p.site,
                showlegend=False,
                customdata=np.column_stack([
                    g.median_ntl, g.sc_pct, g.n_valid
                ]),
                hovertemplate=(
                    f"{p.site}<br>%{{x|%d %b %Y}}"
                    "<br>Four-day RQ NTL %{customdata[0]:.2f}"
                    "<br>SC %{customdata[1]:.0f}%"
                    "<br>Valid pixels %{customdata[2]:.0f}"
                    "<extra></extra>"
                ),
            ),
            row=1, col=col,
        )

    fig.update_xaxes(
        range=[CHARACTERIZATION_START, PRE_EVENT_END],
        tickformat="%b %Y", dtick="M3",
        tickfont=dict(size=15),
        row=1, col=col,
    )

urban_values = poi_box[
    poi_box.site.isin(urban.site)
    & poi_box.window.eq(plot_size)
].median_ntl.dropna()
urban_max = (
    max(5, np.ceil(urban_values.max() * 1.05))
    if len(urban_values) else 5
)
fig.update_yaxes(
    range=[0, urban_max],
    nticks=6,
    title_text="NTL (nW cm⁻² sr⁻¹)",
    tickfont=dict(size=15),
    row=1, col=1,
)

other_values = poi_box[
    poi_box.site.isin(other.site)
    & poi_box.window.eq(plot_size)
].median_ntl.dropna()
other_max = (
    max(1.2, np.ceil(other_values.max() * 1.05))
    if len(other_values) else 1.2
)
raw_ticks = [
    value for value in (0, .25, .5, 1, 2, 5, 10, 15, 20)
    if value <= other_max
]
fig.update_yaxes(
    range=[0, np.log1p(other_max)],
    tickmode="array",
    tickvals=np.log1p(raw_ticks),
    ticktext=[str(value) for value in raw_ticks],
    title_text="NTL (nW cm⁻² sr⁻¹) · log1p axis",
    tickfont=dict(size=15),
    row=1, col=2,
)

style_figure(fig, "", 1840, 920)
fig.update_layout(
    margin=dict(l=90, r=55, t=80, b=290),
    showlegend=False,
    hovermode="closest",
)
fig.update_annotations(font=dict(size=19, color=PLOT_TEXT_COLOR))
add_group_key(fig, urban, 2, 0, .48, "Named class-30 locations")
add_group_key(fig, other, 3, .55, .45, "Individual GHSL pixels")
finish_figure(fig, "poi_four_day_grouped_lines_v3")

The 1×1 and 3×3 site panels show how local aggregation changes both four-day radiance and spatial completeness.

In [ ]:
poi_calendar = pd.date_range(
    CHARACTERIZATION_START,
    CHARACTERIZATION_END,
    freq="D",
)

for size in (1, 3):
    fig = make_subplots(
        rows=4, cols=4,
        row_heights=[.07, .43, .07, .43],
        vertical_spacing=.09,
        horizontal_spacing=.05,
    )
    style_figure(fig, "", 1850, 1080)

    for i, (site, color, symbol) in enumerate(zip(
        poi_sites.site, poi_colors, poi_symbols
    )):
        col = i % 4 + 1
        sc_row = 2*(i // 4) + 1
        ntl_row = sc_row + 1

        d = (
            poi_daily.loc[
                poi_daily.site.eq(site)
                & poi_daily.window.eq(size)
            ]
            .set_index("date")
            .reindex(poi_calendar)
        )
        g = poi_four_day.loc[
            poi_four_day.site.eq(site)
            & poi_four_day.window.eq(size)
        ].sort_values("date")

        fig.add_trace(go.Heatmap(
            x=g.date, y=["SC"],
            z=[g.sc_pct],
            colorscale=SC_GREEN_COLORSCALE,
            zmin=0, zmax=100,
            showscale=(i == 0),
            colorbar=dict(
                title=dict(
                    text="4-day SC (%)",
                    font=dict(size=16),
                ),
                tickfont=dict(size=15),
                len=.15, y=.95,
            ),
            hoverongaps=False,
            hovertemplate=(
                "%{x|%d %b %Y}<br>SC: %{z:.1f}%"
                "<extra></extra>"
            ),
        ), row=sc_row, col=col)

        fig.add_trace(go.Scatter(
            x=d.index,
            y=d.raw_uncapped,
            mode="markers",
            marker=dict(
                size=4,
                color=DNB_COLOR,
                opacity=.40,
            ),
            showlegend=False,
            hovertemplate=(
                "%{x|%d %b %Y}"
                "<br>Daily fresh NTL: %{y:.3f}"
                "<extra></extra>"
            ),
        ), row=ntl_row, col=col)

        legend_id = (
            "legend" if i == 0 else f"legend{i+1}"
        )
        fig.add_trace(go.Scatter(
            x=g.date,
            y=g.ntl,
            mode="lines+markers",
            name=site,
            legend=legend_id,
            line=dict(
                color=color,
                width=2.3,
                shape="hv",
            ),
            marker=dict(
                symbol=symbol,
                size=6,
                color=color,
            ),
            connectgaps=False,
            customdata=g[["sc_pct", "valid_pixels"]],
            hovertemplate=(
                "%{x|%d %b %Y}"
                "<br>Four-day RQ NTL: %{y:.3f}"
                "<br>SC: %{customdata[0]:.1f}%"
                "<br>Valid pixels: %{customdata[1]:.0f}"
                "<extra></extra>"
            ),
        ), row=ntl_row, col=col)

        axis_number = (ntl_row-1)*4 + col
        suffix = (
            "" if axis_number == 1
            else str(axis_number)
        )
        xdomain = fig.layout[f"xaxis{suffix}"].domain
        ydomain = fig.layout[f"yaxis{suffix}"].domain

        fig.update_layout(**{
            legend_id: dict(
                x=xdomain[0],
                y=ydomain[1]+.007,
                xanchor="left",
                yanchor="bottom",
                font=dict(size=15),
                bgcolor="rgba(0,0,0,0)",
            )
        })

        visible_values = np.concatenate([
            d.raw_uncapped.to_numpy(dtype=float),
            g.ntl.to_numpy(dtype=float),
        ])
        visible_values = visible_values[
            np.isfinite(visible_values)
        ]
        ymax = (
            max(.5, np.ceil(
                visible_values.max()*1.05*2
            )/2)
            if visible_values.size else 1.2
        )

        for row in (sc_row, ntl_row):
            fig.update_xaxes(
                range=[
                    CHARACTERIZATION_START,
                    CHARACTERIZATION_END,
                ],
                tickformat="%b %y",
                dtick="M6",
                tickfont=dict(size=14),
                showticklabels=(row == ntl_row),
                title_text=None,
                row=row, col=col,
            )

        fig.update_yaxes(
            showticklabels=False,
            showgrid=False,
            row=sc_row, col=col,
        )
        fig.update_yaxes(
            range=[0, ymax],
            nticks=5,
            tickfont=dict(size=14),
            title_text=(
                "NTL (nW cm⁻² sr⁻¹)"
                if col == 1 else None
            ),
            row=ntl_row, col=col,
        )

    fig.update_layout(
        title=None,
        font=dict(size=16),
        margin=dict(l=95, r=105, t=40, b=65),
    )
    finish_figure(
        fig,
        f"poi_baseline_{size}x{size}",
    )

The site summary and recurrence view describe observed pre-Haiyan variation; missing blocks limit any seasonal interpretation.

In [ ]:
rows = []

for site in poi_sites.site:
    for size in (1, 3, 5):
        g = poi_four_day.loc[
            poi_four_day.site.eq(site)
            & poi_four_day.window.eq(size)
        ]

        for threshold in (10., 50., 60.):
            observed = g.loc[
                g.sc_pct.ge(threshold)
                & g.valid_pixels.gt(0)
            ]
            ntl = observed.rq_median.dropna()
            relative = observed.reference_pct.dropna()

            rows.append(dict(
                site=site,
                window=f"{size}×{size}",
                sc_threshold_pct=threshold,
                fixed_pixels=len(
                    poi_fixed[(site, size)]
                ),
                observed_blocks=len(ntl),
                missing_blocks=len(g)-len(ntl),
                observed_zero_blocks=int(
                    ntl.eq(0).sum()
                ),
                retained_pct=100*len(ntl)/len(g),
                median_ntl=ntl.median(),
                p10_ntl=ntl.quantile(.10),
                p90_ntl=ntl.quantile(.90),
                mad_ntl=(
                    ntl-ntl.median()
                ).abs().median(),
                mad_reference_pct=(
                    relative-relative.median()
                ).abs().median(),
            ))

poi_metrics = pd.DataFrame(rows)

print("Named pixel · primary SC ≥10%")
display(
    poi_metrics.loc[
        poi_metrics.window.eq("1×1")
        & poi_metrics.sc_threshold_pct.eq(10)
    ].round(2)
)

print("How 3×3 and 5×5 change the signal")
display(
    poi_metrics.loc[
        poi_metrics.window.isin(["3×3", "5×5"])
        & poi_metrics.sc_threshold_pct.eq(10)
    ].round(2)
)

print("SC threshold sensitivity")
display(
    poi_metrics.loc[
        poi_metrics.window.eq("3×3"),
        [
            "site", "sc_threshold_pct",
            "observed_blocks", "retained_pct",
            "median_ntl", "mad_ntl",
            "mad_reference_pct",
        ],
    ].round(2)
)

# Same-time-of-year comparison of observed 3×3 blocks.
# This is an annual-recurrence diagnostic, not gap filling.
fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=poi_sites.site.tolist(),
    horizontal_spacing=.055,
    vertical_spacing=.12,
)
recurrence_rows = []

for i, site in enumerate(poi_sites.site):
    row, col = divmod(i, 4)
    row += 1
    col += 1

    g = poi_four_day.loc[
        poi_four_day.site.eq(site)
        & poi_four_day.window.eq(3)
    ].copy()

    g = g.loc[
        ~(
            g.date.dt.month.eq(2)
            & g.date.dt.day.eq(29)
        )
    ]
    g["year"] = g.date.dt.year
    g["season_day"] = (
        g.date.dt.dayofyear
        - (
            g.date.dt.is_leap_year
            & g.date.dt.dayofyear.gt(60)
        ).astype(int)
    )

    for year, color in [
        (2012, DNB_COLOR),
        (2013, "#E69F00"),
    ]:
        yearly = g.loc[g.year.eq(year)]
        fig.add_trace(go.Scatter(
            x=yearly.season_day,
            y=yearly.ntl,
            mode="lines+markers",
            name=str(year),
            legendgroup=str(year),
            showlegend=(i == 0),
            line=dict(color=color, width=1.7),
            marker=dict(size=5),
            connectgaps=False,
            customdata=yearly[["sc_pct"]],
            hovertemplate=(
                f"{year} · day %{{x}}"
                "<br>RQ NTL: %{y:.3f}"
                "<br>SC: %{customdata[0]:.1f}%"
                "<extra></extra>"
            ),
        ), row=row, col=col)

    for threshold in (10., 50., 60.):
        observed = g.loc[
            g.sc_pct.ge(threshold)
            & g.valid_pixels.gt(0)
            & g.rq_median.notna()
        ]

        a = observed.loc[
            observed.year.eq(2012),
            ["season_day", "rq_median"],
        ].sort_values("season_day")
        b = observed.loc[
            observed.year.eq(2013),
            ["season_day", "rq_median"],
        ].sort_values("season_day")

        matched = pd.merge_asof(
            a, b,
            on="season_day",
            direction="nearest",
            tolerance=2,
            suffixes=("_2012", "_2013"),
        ).dropna()

        correlation = (
            matched.rq_median_2012.corr(
                matched.rq_median_2013
            )
            if len(matched) >= 20
            and matched.rq_median_2012.std() > 0
            and matched.rq_median_2013.std() > 0
            else np.nan
        )

        recurrence_rows.append(dict(
            site=site,
            sc_threshold_pct=threshold,
            matched_pairs=len(matched),
            year_to_year_correlation=correlation,
        ))

style_figure(fig, "", 1850, 900)
fig.update_layout(
    title=None,
    font=dict(size=16),
    margin=dict(l=75, r=35, t=55, b=110),
    legend=dict(
        orientation="h",
        x=0, y=-.12,
        font=dict(size=17),
    ),
)
fig.update_annotations(
    font=dict(size=16, color=PLOT_TEXT_COLOR)
)
fig.update_xaxes(
    tickvals=[1, 60, 121, 182, 244, 305],
    ticktext=["Jan", "Mar", "May", "Jul", "Sep", "Nov"],
    tickfont=dict(size=14),
)
fig.update_yaxes(tickfont=dict(size=14))
finish_figure(fig, "poi_annual_recurrence")

recurrence = pd.DataFrame(recurrence_rows)
display(recurrence.round(2))

The closing comparison is centred on each site's **pre-Haiyan median**. Post-Haiyan departures do not redefine that baseline. Four-day gaps remain missing.

In [ ]:
# Selected 1×1 pixels: pre- and post-Haiyan four-day RQ NTL.
# The zero line and displayed median use PRE-EVENT observations only.

plot_size = 1
POST_DAYS = 365
plot_end = min(
    pd.Timestamp(dates.max()),
    EVENT_DATE + pd.Timedelta(days=POST_DAYS),
)

urban = site_df.loc[site_df.ghsl.eq(30)].copy()
other = site_df.loc[site_df.ghsl.ne(30)].copy()

if len(urban) != 8 or len(other) < 4:
    raise ValueError(
        f"Expected 8 urban sites and at least 4 comparison pixels; "
        f"found {len(urban)} and {len(other)}."
    )

# The existing poi_box contains pre-Haiyan blocks only.
# Extract matching post-Haiyan 1×1 blocks using the same RQ method.
post_rows = []
last_post_block = int(
    (plot_end - EVENT_DATE).days // AGGREGATION_DAYS
)

for block in tqdm(
    range(0, last_post_block + 1),
    desc="Post-Haiyan 1×1 pixels",
):
    start = EVENT_DATE + pd.Timedelta(
        days=block * AGGREGATION_DAYS
    )
    end = start + pd.Timedelta(days=AGGREGATION_DAYS - 1)
    if end > plot_end:
        continue

    midpoint = start + pd.Timedelta(
        days=(AGGREGATION_DAYS - 1) / 2
    )
    indices = np.flatnonzero(day_blocks == block)

    if len(indices):
        _, capped, _ = read_days(indices)
        composite = median(capped, axis=0)
    else:
        composite = np.full(
            len(land_pixels), np.nan, dtype=np.float32
        )

    for site in site_df.site:
        support = poi_fixed[(site, plot_size)]
        values = composite[support]
        n_fixed = len(support)
        n_valid = int(np.isfinite(values).sum())
        sc_pct = (
            100 * n_valid / n_fixed if n_fixed else np.nan
        )
        observed = (
            n_valid > 0
            and sc_pct >= SPATIAL_COMPLETENESS_PCT
        )

        post_rows.append(dict(
            site=site,
            window=plot_size,
            date=midpoint,
            date_start=start,
            date_end=end,
            median_ntl=median(values) if observed else np.nan,
            sc_pct=sc_pct,
            n_valid=n_valid,
            n_fixed=n_fixed,
            observed=observed,
        ))

poi_plot = pd.concat(
    [
        poi_box.loc[poi_box.date.lt(EVENT_DATE)],
        pd.DataFrame(post_rows),
    ],
    ignore_index=True,
)

# Prepare series and fix each zero line to its pre-Haiyan median.
site_series = {}
for p in pd.concat([urban, other]).itertuples():
    g = poi_plot.loc[
        poi_plot.site.eq(p.site)
        & poi_plot.window.eq(plot_size)
        & poi_plot.date.le(plot_end)
    ].sort_values("date")

    pre = g.loc[g.date_end.lt(EVENT_DATE)]
    baseline_median = pre.median_ntl.median()
    if not np.isfinite(baseline_median):
        raise ValueError(f"No pre-Haiyan median for {p.site}.")

    relative_ntl = (
        np.log1p(g.median_ntl.astype(float))
        - np.log1p(baseline_median)
    )
    site_series[p.site] = (g, relative_ntl, baseline_median)

all_values = np.concatenate([
    relative.dropna().to_numpy()
    for _, relative, _ in site_series.values()
])
limit = max(
    .5,
    np.ceil(np.max(np.abs(all_values)) * 1.05 * 2) / 2,
)
tick = limit / 2
y_ticks = [-tick, 0, tick]
y_tick_labels = [f"−{tick:g}", "0", f"+{tick:g}"]

In [ ]:
fig = go.Figure()
style_figure(fig, "", 1600, 900)

groups = [
    (urban, (.205, .505), "Urban-centre sites"),
    (other, (.665, .965), "Other settlement pixels"),
]
top, bottom, lane_gap = .94, .08, .009
axis_number = 0

for group, x_domain, heading in groups:
    fig.add_annotation(
        x=(x_domain[0] + x_domain[1]) / 2,
        y=.985,
        xref="paper", yref="paper",
        xanchor="center",
        text=heading,
        showarrow=False,
        font=dict(size=21, color=PLOT_TEXT_COLOR),
    )
    fig.add_annotation(
        x=x_domain[1], y=.99,
        xref="paper", yref="paper",
        text="Pre-Haiyan<br>median NTL",
        xanchor="left",
        showarrow=False,
        font=dict(size=13, color=PLOT_TEXT_COLOR),
    )

    lane_height = (
        top - bottom - (len(group) - 1) * lane_gap
    ) / len(group)

    for row, p in enumerate(group.itertuples()):
        axis_number += 1
        xref = "x" if axis_number == 1 else f"x{axis_number}"
        yref = "y" if axis_number == 1 else f"y{axis_number}"
        xaxis_key = (
            "xaxis" if axis_number == 1
            else f"xaxis{axis_number}"
        )
        yaxis_key = (
            "yaxis" if axis_number == 1
            else f"yaxis{axis_number}"
        )

        y_high = top - row * (lane_height + lane_gap)
        y_low = y_high - lane_height
        g, relative_ntl, baseline_median = site_series[p.site]

        color = (
            p.color if p.ghsl == 30
            else {
                21: "#B49B00",
                12: "#79A83F",
                11: "#98B847",
            }.get(
                int(p.ghsl),
                GHSL_CLASS_COLORS[int(p.ghsl)],
            )
        )

        fig.add_trace(go.Scatter(
            x=g.date,
            y=relative_ntl,
            xaxis=xref,
            yaxis=yref,
            mode="lines",
            line=dict(color=color, width=1.7),
            connectgaps=False,
            name=p.site,
            showlegend=False,
            customdata=np.column_stack([
                g.median_ntl, g.sc_pct, g.n_valid
            ]),
            hovertemplate=(
                f"{p.site}<br>%{{x|%d %b %Y}}"
                "<br>RQ NTL (4D) %{customdata[0]:.2f}"
                "<br>SC %{customdata[1]:.0f}%"
                "<br>Valid pixels %{customdata[2]:.0f}"
                "<extra></extra>"
            ),
        ))

        fig.update_layout(**{
            xaxis_key: dict(
                domain=list(x_domain),
                anchor=yref,
                range=[CHARACTERIZATION_START, plot_end],
                dtick="M6",
                tickformat="%b %Y",
                tickfont=dict(size=13),
                showticklabels=(row == len(group) - 1),
                showgrid=True,
                gridcolor="rgba(36,59,90,.13)",
                minor=dict(
                    dtick="M1",
                    showgrid=True,
                    gridcolor="rgba(36,59,90,.06)",
                ),
                zeroline=False,
            ),
            yaxis_key: dict(
                domain=[y_low, y_high],
                anchor=xref,
                range=[-limit, limit],
                tickmode="array",
                tickvals=y_ticks,
                ticktext=y_tick_labels,
                tickfont=dict(size=12),
                ticks="outside",
                ticklen=3,
                showgrid=False,
                zeroline=False,
            ),
        })

        # The same pre-Haiyan median anchors the entire timeline.
        fig.add_shape(
            type="line",
            xref=f"{xref} domain",
            yref=yref,
            x0=0, x1=1,
            y0=0, y1=0,
            line=dict(color="rgba(36,59,90,.50)", width=1),
            layer="below",
        )

        fig.add_shape(
            type="line",
            xref=xref,
            yref="paper",
            x0=EVENT_DATE.to_pydatetime(),
            x1=EVENT_DATE.to_pydatetime(),
            y0=y_low,
            y1=y_high,
            line=dict(
                color=EVENT_COLOR,
                width=1.2,
                dash="dash",
            ),
            layer="above",
        )

        fig.add_annotation(
            x=x_domain[0] - .038,
            y=(y_low + y_high) / 2,
            xref="paper", yref="paper",
            text=p.site,
            xanchor="right",
            yanchor="middle",
            showarrow=False,
            font=dict(size=15, color=PLOT_TEXT_COLOR),
        )

        # Pre-Haiyan median: same size as the site name, coloured to match its line.
        fig.add_annotation(
            x=x_domain[1] + .006,
            y=0,
            xref="paper", yref=yref,
            text=f"{baseline_median:.2f}",
            xanchor="left",
            yanchor="middle",
            showarrow=False,
            font=dict(size=15, color=color),
        )

fig.update_layout(
    title=None,
    margin=dict(l=20, r=20, t=20, b=20),
    showlegend=False,
    hovermode="closest",
)
finish_figure(fig, "poi_four_day_pre_post_median_centered")